In [ ]:
# Imports
import os
os.environ['OMP_NUM_THREADS'] = '1'

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union, Any
import json
import pickle
import warnings
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, mannwhitneyu, gaussian_kde, ttest_ind, f_oneway
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Statistical packages
import pingouin as pg
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

In [ ]:
# CORRECTED MOTOR LEARNING ANALYSIS PIPELINE
# ============================================
# This corrected version fixes structural issues and improves organization

# ==============================================================================
# CONFIGURATION AND DATA STRUCTURES
# ==============================================================================

@dataclass
class AnalysisConfig:
    """Centralized configuration for all analysis parameters."""
    
    # Directories
    base_output_dir: Path = Path('motor_learning_output')
    
    # Thresholds
    min_complete_strides: int = 20
    motor_noise_strides: int = 20
    motor_noise_threshold: float = 0.3
    success_rate_threshold: float = 0.68
    target_size_threshold: float = 0.31
    max_strides_threshold: int = 415
    
    # Visualization
    figure_dpi: int = 300
    alpha_level: float = 0.05
    age_bins: List[int] = field(default_factory=lambda: [7, 10, 13, 16, 18])
    age_labels: List[str] = field(default_factory=lambda: ['7-10', '10-13', '13-16', '16-18'])
    
    # Trial mappings
    trial_type_mapping: Dict[str, str] = field(default_factory=lambda: {
        'primer': 'vis1',
        'trial': 'invis',
        'vis': 'vis2',
        'pref': 'pref'
    })
    
    def __post_init__(self):
        """Create directory structure."""
        self.figures_dir = self.base_output_dir / 'figures'
        self.individual_plots_dir = self.figures_dir / 'individual_plots'
        self.population_plots_dir = self.figures_dir / 'population_plots'
        self.statistical_plots_dir = self.figures_dir / 'statistical_plots'
        self.reports_dir = self.base_output_dir / 'reports'
        self.exports_dir = self.base_output_dir / 'exports'
        self.processed_data_dir = self.base_output_dir / 'processed_data'
        
        # Create all directories
        for directory in [self.figures_dir, self.individual_plots_dir, 
                         self.population_plots_dir, self.statistical_plots_dir,
                         self.reports_dir, self.exports_dir, self.processed_data_dir]:
            directory.mkdir(parents=True, exist_ok=True)
        
        self.processed_data_file = self.processed_data_dir / 'processed_data.pkl'

@dataclass
class SubjectData:
    """Container for individual subject data."""
    subject_id: str
    metadata: Dict[str, Any]
    trial_data: Dict[str, Dict[str, Any]]
    
    @property
    def age(self) -> float:
        """Get age in years."""
        return self.metadata.get('age_months', np.nan) / 12

@dataclass
class TrialData:
    """Container for trial data."""
    data: pd.DataFrame
    anomalies: Dict[int, List[str]]
    trial_type: str
    
    @property
    def is_valid(self) -> bool:
        """Check if trial data is valid."""
        return self.data is not None and not self.data.empty

# ==============================================================================
# BASE CLASSES
# ==============================================================================

class BaseProcessor(ABC):
    """Base class for all data processors."""
    
    def __init__(self, config: AnalysisConfig, debug: bool = True):
        self.config = config
        self.debug = debug
    
    def log(self, message: str, level: str = "info"):
        """Logging helper."""
        if self.debug:
            symbols = {"info": "📊", "warning": "⚠️", "error": "❌", "success": "✓"}
            print(f"{symbols.get(level, '•')} {message}")

class BaseVisualizer(ABC):
    """Base class for all visualizers."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.colors = {
            'primary': '#667eea',
            'secondary': '#764ba2',
            'success': '#28a745',
            'warning': '#ffc107',
            'danger': '#dc3545',
            'vis1': '#1f77b4',
            'invis': '#ff7f0e',
            'vis2': '#2ca02c'
        }
    
    def save_figure(self, fig: plt.Figure, filename: str, subdir: str = 'general'):
        """Save figure to appropriate directory."""
        if subdir == 'individual':
            save_path = self.config.individual_plots_dir / filename
        elif subdir == 'population':
            save_path = self.config.population_plots_dir / filename
        elif subdir == 'statistical':
            save_path = self.config.statistical_plots_dir / filename
        else:
            save_path = self.config.figures_dir / filename
        
        fig.savefig(save_path, dpi=self.config.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        return save_path
    
    def add_trendline(self, ax, x, y):
        """Add trendline with correlation to plot."""
        x_clean = pd.to_numeric(x, errors='coerce')
        y_clean = pd.to_numeric(y, errors='coerce')
        valid = x_clean.notna() & y_clean.notna()
        
        if valid.sum() < 2:
            return
        
        x_vals = x_clean[valid]
        y_vals = y_clean[valid]
        
        try:
            coeffs = np.polyfit(x_vals, y_vals, 1)
            trendline = np.poly1d(coeffs)
            r, p = pearsonr(x_vals, y_vals)
            
            ax.plot(x_vals, trendline(x_vals), 'r--', alpha=0.8, linewidth=2)
            ax.text(0.05, 0.95, f'r² = {r**2:.3f}\np = {p:.3f}\nn = {len(x_vals)}', 
                   transform=ax.transAxes,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   verticalalignment='top', fontsize=10)
        except Exception:
            pass

# ==============================================================================
# DATA PROCESSING COMPONENTS
# ==============================================================================

class DataValidator:
    """Handles data validation and anomaly detection."""
    
    @staticmethod
    def validate_dataframe(df: pd.DataFrame, required_cols: List[str] = None) -> bool:
        """Validate dataframe has required columns and data."""
        if df is None or df.empty:
            return False
        if required_cols and not all(col in df.columns for col in required_cols):
            return False
        return True
    
    @staticmethod
    def detect_anomalies(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Detect and flag anomalies in stride data."""
        if df is None or df.empty:
            return df, {}
        
        df = df.copy()
        df['Anomalous'] = False
        anomalies = {}
        
        # Time-based anomalies
        time_col = next((col for col in ['Time', 'Timestamp', 'Time (s)'] 
                        if col in df.columns), None)
        if time_col:
            df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
            time_diff = df[time_col].diff()
            jump_mask = time_diff > time_diff.quantile(0.99) * 5
            
            for idx in df.index[jump_mask.fillna(False)]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('time_jump')
        
        # Sum of gains and steps anomalies
        if 'Sum of gains and steps' in df.columns:
            high_mask = df['Sum of gains and steps'] > 4
            zero_mask = df['Sum of gains and steps'] == 0
            
            for idx in df.index[high_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_high')
            
            for idx in df.index[zero_mask]:
                df.at[idx, 'Anomalous'] = True
                anomalies.setdefault(idx, []).append('sum_gain_step_zero')
        
        return df, anomalies

class FileLoader:
    """Handles file loading operations."""
    
    @staticmethod
    def load_file(file_path: Path) -> Optional[pd.DataFrame]:
        """Load and validate a single data file."""
        try:
            df = pd.read_csv(file_path, sep='\t')
            
            # Basic cleaning
            if 'Stride Number' in df.columns:
                df['Stride Number'] = pd.to_numeric(df['Stride Number'], errors='coerce')
                df = df.dropna(subset=['Stride Number'])
                df = df.drop_duplicates(subset=['Stride Number'])
                df = df.sort_values('Stride Number')
            
            return df if not df.empty else None
            
        except Exception:
            return None

class TrialProcessor(BaseProcessor):
    """Processes trial data files."""
    
    def process_trial_files(self, subject_dir: Path, trial_prefix: str) -> Optional[pd.DataFrame]:
        """Find and combine trial files for a given trial type."""
        all_files = sorted(subject_dir.glob(f"{trial_prefix}*.txt"))
        
        if not all_files:
            return None
        
        # Special handling for preference trials
        if trial_prefix == 'pref':
            largest_file = max(all_files, key=lambda f: f.stat().st_size)
            return FileLoader.load_file(largest_file)
        
        # Single file case
        if len(all_files) == 1:
            return FileLoader.load_file(all_files[0])
        
        # Multiple files - combine them
        return self._combine_trial_fragments(all_files)
    
    def _combine_trial_fragments(self, files: List[Path]) -> Optional[pd.DataFrame]:
        """Combine multiple trial fragments intelligently."""
        dfs = []
        for f in files:
            df = FileLoader.load_file(f)
            if df is not None:
                dfs.append(df)
        
        if not dfs:
            return None
        
        combined = pd.concat(dfs, ignore_index=True)
        
        if 'Stride Number' in combined.columns:
            combined = combined.sort_values('Stride Number')
            combined = combined.drop_duplicates('Stride Number')
        
        return combined

# ==============================================================================
# METRICS CALCULATION
# ==============================================================================

class MetricsEngine:
    """Core metrics calculation engine."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config

    def calculate_stride_change_distributions(self, trial_data: Dict, trial_type: str) -> Dict:
        """
        Calculate stride length change distributions after success and failure for a participant.
        
        Returns only the raw distributions, excluding success clamps and baseline blocks.
        """
        
        metrics = {}
        
        if not trial_data or trial_data['data'] is None:
            return metrics
        
        df = trial_data['data'].copy()
        
        # Check required columns
        required_cols = ['Stride Number', 'Success', 'Sum of gains and steps']
        if not all(col in df.columns for col in required_cols):
            return metrics
        
        # Sort by stride number
        df = df.sort_values('Stride Number').reset_index(drop=True)
        
        try:
            # Exclude success clamps and baseline blocks
            excluded_mask = self._identify_excluded_periods(df, trial_type)
            
            # Get the included data
            included_df = df[~excluded_mask].copy()
            
            if len(included_df) < 10:  # Need minimum data
                return metrics
            
            # Calculate stride changes
            included_df = included_df.sort_values('Stride Number').reset_index(drop=True)
            included_df['stride_change'] = included_df['Sum of gains and steps'].diff()
            
            # Remove first row (no previous stride to compare to)
            analysis_df = included_df.iloc[1:].copy()
            
            if len(analysis_df) < 5:
                return metrics
            
            # Get previous trial outcome
            analysis_df['prev_success'] = included_df['Success'].iloc[:-1].values
            
            # Separate changes after success vs failure
            after_success = analysis_df[analysis_df['prev_success'] == 1]['stride_change'].dropna()
            after_failure = analysis_df[analysis_df['prev_success'] == 0]['stride_change'].dropna()
            
            # Store only the raw distributions
            metrics[f'{trial_type}_stride_changes_after_success'] = after_success.tolist()
            metrics[f'{trial_type}_stride_changes_after_failure'] = after_failure.tolist()
            
        except Exception as e:
            print(f"Error calculating stride changes for {trial_type}: {e}")
        
        return metrics
    
    def _identify_excluded_periods(self, df: pd.DataFrame, trial_type: str) -> pd.Series:
        """
        Identify periods to exclude from stride change analysis.
        
        Excludes:
        1. Success clamps: max target size AND const = 2
        2. Baseline block in invisible trials: const = 2 at beginning
        """
        
        excluded_mask = pd.Series(False, index=df.index)
        
        # Check if we have the required columns for exclusion logic
        if not all(col in df.columns for col in ['Target size', 'Constant']):
            return excluded_mask
        
        # 1. Exclude success clamps (max target size with const = 2)
        max_target_size = df['Target size'].max()
        target_tolerance = 0.01
        
        success_clamp_mask = (
            (df['Target size'] >= max_target_size - target_tolerance) & 
            (np.abs(df['Constant'] - 2.0) < target_tolerance)
        )
        
        excluded_mask |= success_clamp_mask
        
        # 2. For invisible trials, exclude baseline block at beginning
        if trial_type == 'invis':
            # Find the initial baseline period (const = 2 at the beginning)
            baseline_mask = np.abs(df['Constant'] - 2.0) < target_tolerance
            
            if baseline_mask.any():
                # Find the first contiguous block of baseline strides
                baseline_indices = df.index[baseline_mask].tolist()
                
                # Check if baseline starts from the beginning
                if baseline_indices and baseline_indices[0] <= 5:  # Allow some tolerance for start
                    # Find where the first baseline block ends
                    consecutive_baseline = []
                    for i, idx in enumerate(baseline_indices):
                        if i == 0 or idx == baseline_indices[i-1] + 1:
                            consecutive_baseline.append(idx)
                        else:
                            break
                    
                    # Exclude the first baseline block
                    if len(consecutive_baseline) >= 10:  # Only if it's substantial
                        for idx in consecutive_baseline:
                            excluded_mask.iloc[idx] = True
        
        return excluded_mask
    
    def calculate_period_metrics(self, period_data: pd.DataFrame, 
                               trial_type: str, condition: str) -> Dict:
        """Calculate all metrics for a specific period, including the new learning metric."""
        metrics = {}
        
        # Success rate
        metrics[f'{trial_type}_sr_{condition}_const'] = period_data['Success'].mean()
        
        # Stride metrics
        if 'Sum of gains and steps' in period_data.columns:
            sogs = period_data['Sum of gains and steps']
            metrics[f'{trial_type}_sd_{condition}_const'] = sogs.std()
            metrics[f'{trial_type}_msl_{condition}_const'] = sogs.mean()
            
            # NEW: Learning metric calculation
            if 'Constant' in period_data.columns:
                const_value = period_data['Constant'].iloc[0] if not period_data['Constant'].empty else None
                avg_stride = sogs.mean()  # This is the mean stride length over the period
                preferred_stride = 2.0  # Normalized preferred stride length
                
                if const_value is not None and not pd.isna(const_value):
                    # Learning = (avg_stride - preferred_stride) / (const - preferred_stride)
                    denominator = const_value - preferred_stride
                    if abs(denominator) > 1e-6:  # Avoid division by zero
                        learning_value = (avg_stride - preferred_stride) / denominator
                        metrics[f'{trial_type}_learning_{condition}_const'] = learning_value
                    else:
                        # If const equals preferred_stride, learning is undefined
                        metrics[f'{trial_type}_learning_{condition}_const'] = np.nan
                else:
                    metrics[f'{trial_type}_learning_{condition}_const'] = np.nan
                
                # Original error calculation
                if 'Constant' in period_data.columns:
                    metrics[f'{trial_type}_error_{condition}_const'] = (sogs - period_data['Constant']).mean()
        
        # Asymmetry
        if all(col in period_data.columns for col in ['Right step length', 'Left step length']):
            asymmetry = self._calculate_asymmetry(
                period_data['Right step length'], 
                period_data['Left step length']
            )
            if asymmetry is not None:
                metrics[f'{trial_type}_asymmetry_{condition}_const'] = asymmetry
        
        # Strides between successes
        strides_between = self._calculate_strides_between_successes(period_data)
        if strides_between is not None:
            metrics[f'{trial_type}_strides_between_success_{condition}_const'] = strides_between
        
        return metrics
    
    def calculate_preference_metrics(self, pref_df: pd.DataFrame) -> Dict:
        """Calculate metrics from preference trial."""
        metrics = {'mot_noise': None, 'pref_asymmetry': None}
        
        if pref_df is None or pref_df.empty:
            return metrics
        
        if not all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
            return metrics
        
        # Get non-zero steps
        right_steps = pref_df['Right step length']
        left_steps = pref_df['Left step length']
        
        right_clean = right_steps[(right_steps != 0) & (right_steps.notna())]
        left_clean = left_steps[(left_steps != 0) & (left_steps.notna())]
        
        min_required = self.config.motor_noise_strides
        if len(right_clean) < min_required or len(left_clean) < min_required:
            return metrics
        
        # Calculate motor noise
        final_right = right_clean.iloc[-1]
        final_left = left_clean.iloc[-1]
        
        if final_right <= 0 or final_left <= 0:
            return metrics
        
        # Normalize steps
        norm_right = right_clean / final_right
        norm_left = left_clean / final_left
        
        min_length = min(len(norm_right), len(norm_left))
        if min_length < min_required:
            return metrics
        
        sum_steps = norm_right.iloc[:min_length] + norm_left.iloc[:min_length]
        
        # Motor noise from last N points
        if len(sum_steps) >= self.config.motor_noise_strides:
            noise = sum_steps.tail(self.config.motor_noise_strides).std()
            if not pd.isna(noise) and noise > 0:
                metrics['mot_noise'] = noise
        
        # Calculate asymmetry
        if len(right_clean) >= self.config.motor_noise_strides and len(left_clean) >= self.config.motor_noise_strides:
            last_n_right = right_clean.tail(self.config.motor_noise_strides) / final_right
            last_n_left = left_clean.tail(self.config.motor_noise_strides) / final_left
            
            asymmetry = self._calculate_asymmetry(last_n_right.values, last_n_left.values)
            if asymmetry is not None:
                metrics['pref_asymmetry'] = asymmetry
        
        return metrics
    
    def _calculate_asymmetry(self, right_values, left_values) -> Optional[float]:
        """Calculate step length asymmetry."""
        denominator = right_values + left_values
        valid_mask = denominator != 0
        
        if not valid_mask.any():
            return None
        
        asymmetry_vals = np.abs((right_values - left_values) / denominator)[valid_mask]
        return np.mean(asymmetry_vals) if len(asymmetry_vals) > 0 else None
    
    def _calculate_strides_between_successes(self, df: pd.DataFrame) -> Optional[float]:
        """Calculate average strides between successful trials."""
        if df is None or 'Success' not in df.columns:
            return None
        
        df = df.reset_index(drop=True)
        success_positions = df.index[df['Success'] == 1].tolist()
        
        if len(success_positions) < 2:
            return None
        
        return np.mean(np.diff(success_positions))


# ==============================================================================
# DATA MANAGEMENT
# ==============================================================================

class DataManager(BaseProcessor):
    """Manages all data loading and processing operations."""
    
    def __init__(self, metadata_path: str, data_root_dir: str, 
                 config: AnalysisConfig, force_reprocess: bool = False, 
                 debug: bool = True):
        super().__init__(config, debug)
        self.metadata_path = metadata_path
        self.data_root_dir = Path(data_root_dir)
        self.trial_processor = TrialProcessor(config, debug)
        self.metrics_engine = MetricsEngine(config)
        
        self.subjects: Dict[str, SubjectData] = {}
        self.metadata: pd.DataFrame = None
        
        if not force_reprocess and config.processed_data_file.exists():
            self._load_processed_data()
        else:
            self._process_all_data()
            self._save_processed_data()
    
    def _load_processed_data(self):
        """Load previously processed data."""
        try:
            with open(self.config.processed_data_file, 'rb') as f:
                saved_data = pickle.load(f)
                
            # Convert to SubjectData objects
            for subject_id, data in saved_data.items():
                self.subjects[subject_id] = SubjectData(
                    subject_id=subject_id,
                    metadata=data['metadata'],
                    trial_data=data['trial_data']
                )
            
            # Rebuild metadata DataFrame
            self.metadata = pd.DataFrame.from_dict(
                {subj: data.metadata for subj, data in self.subjects.items()}, 
                orient='index'
            )
            self.log(f"Loaded {len(self.subjects)} subjects from cache", "success")
        except Exception as e:
            self.log(f"Failed to load cached data: {e}", "warning")
            self._process_all_data()
            self._save_processed_data()
    
    def _save_processed_data(self):
        """Save processed data."""
        try:
            # Convert to serializable format
            save_data = {}
            for subject_id, subject in self.subjects.items():
                save_data[subject_id] = {
                    'metadata': subject.metadata,
                    'trial_data': subject.trial_data
                }
            
            with open(self.config.processed_data_file, 'wb') as f:
                pickle.dump(save_data, f)
            self.log(f"Saved processed data to {self.config.processed_data_file}", "success")
        except Exception as e:
            self.log(f"Failed to save processed data: {e}", "error")
    
    def _process_all_data(self):
        """Process all subject data."""
        self._load_metadata()
        total_subjects = len(self.metadata)
        
        self.log(f"Processing {total_subjects} subjects...")
        
        for i, (_, row) in enumerate(self.metadata.iterrows(), 1):
            subject_id = row['ID']
            if self.debug and i % 10 == 0:
                self.log(f"Progress: {i}/{total_subjects}")
            
            subject_data = self._process_subject(subject_id, row)
            if subject_data:
                self.subjects[subject_id] = subject_data
    
    def _load_metadata(self):
        """Load and clean metadata."""
        self.metadata = pd.read_csv(self.metadata_path)
        self.metadata['DOB'] = pd.to_datetime(self.metadata['DOB'], errors='coerce')
        self.metadata['Session Date'] = pd.to_datetime(self.metadata['Session Date'], errors='coerce')
        self.metadata = self.metadata.dropna(subset=['ID', 'age_months'])
    
    def _process_subject(self, subject_id: str, metadata_row: pd.Series) -> Optional[SubjectData]:
        """Process data for a single subject."""
        subject_dir = self.data_root_dir / subject_id
        if not subject_dir.exists():
            return None
        
        trial_data = {}
        
        for original_type, mapped_type in self.config.trial_type_mapping.items():
            df = self.trial_processor.process_trial_files(subject_dir, original_type)
            
            if df is not None:
                # Process the data
                if original_type != 'pref':
                    df, anomalies = self._process_trial_data(df)
                else:
                    df = df.drop_duplicates(subset='Left heel strike', keep='last')
                    anomalies = {}
                
                trial_data[mapped_type] = {
                    'data': df,
                    'anomalies': anomalies
                }
        
        if not trial_data:
            return None
        
        return SubjectData(
            subject_id=subject_id,
            metadata=metadata_row.to_dict(),
            trial_data=trial_data
        )
    
    def _process_trial_data(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
        """Process trial data and calculate derived metrics."""
        if df is None or df.empty:
            return None, {}
        
        # Validate required columns
        required_cols = ['Stride Number', 'Success', 'Upper bound success', 
                        'Lower bound success', 'Constant']
        if not all(col in df.columns for col in required_cols):
            return None, {}
        
        # Process trial data
        df = df.sort_values('Stride Number')
        df['Target size'] = df['Upper bound success'] - df['Lower bound success']
        df = df.drop_duplicates(subset='Stride Number', keep='last')
        
        # Scale sum of gains and steps
        if 'Sum of gains and steps' in df.columns:
            df['Sum of gains and steps'] = 1.5 * df['Sum of gains and steps']
        
        # Detect anomalies
        df, anomalies = DataValidator.detect_anomalies(df)
        
        return df, anomalies
    
    def get_trial_data(self, subject_id: str, trial_type: str) -> Optional[pd.DataFrame]:
        """Get trial data for a specific subject and trial type."""
        if subject_id not in self.subjects:
            return None
        
        trial_dict = self.subjects[subject_id].trial_data.get(trial_type)
        return trial_dict['data'] if trial_dict else None
    
    def calculate_metrics(self) -> pd.DataFrame:
        """Calculate metrics for all subjects - ENHANCED WITH SIMPLIFIED RETENTION."""
        results = []
        
        self.log(f"Calculating metrics for {len(self.subjects)} subjects...")
        
        for subject_id, subject in self.subjects.items():
            result = self._calculate_subject_metrics(subject)
            if result:
                results.append(result)
        
        if not results:
            self.log("No valid metrics calculated!", level="error")
            return pd.DataFrame()
        
        df = pd.DataFrame(results).infer_objects()
        
        # CALCULATE RETENTION METRICS AFTER ALL OTHER METRICS
        df = self._calculate_retention_for_all_subjects(df)
        
        self.log(f"Successfully calculated metrics for {len(df)} subjects", level="success")
        return df
    
    def _calculate_retention_for_all_subjects(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate retention metrics for all subjects using existing metrics.
        This is much simpler than the original approach.
        """
        
        # Initialize retention columns
        df['invis_retention_clamp1_percent'] = np.nan
        df['invis_retention_clamp2_percent'] = np.nan
        df['invis_retention_overall_percent'] = np.nan
        
        for idx, row in df.iterrows():
            subject_id = row['ID']
            
            # Only calculate for subjects with invisible trial data
            if subject_id not in self.subjects:
                continue
                
            subject = self.subjects[subject_id]
            invis_trial = subject.trial_data.get('invis')
            
            if not invis_trial or invis_trial['data'] is None:
                continue
                
            # Calculate retention using the simpler approach
            retention_metrics = self._calculate_subject_retention(subject_id, invis_trial['data'])
            
            # Update the dataframe with retention metrics
            for metric_name, value in retention_metrics.items():
                if metric_name in df.columns:
                    df.at[idx, metric_name] = value
        
        return df
    

    def _calculate_subject_retention(self, subject_id: str, invis_df: pd.DataFrame) -> Dict:
        """
        Calculate retention for a single subject using a simplified approach.
        FIXED: Properly exclude baseline block and identify only true success clamps.
        """
        
        metrics = {
            'invis_retention_clamp1_percent': np.nan,
            'invis_retention_clamp2_percent': np.nan, 
            'invis_retention_overall_percent': np.nan
        }
        
        if invis_df is None or invis_df.empty:
            return metrics
            
        required_cols = ['Target size', 'Constant', 'Sum of gains and steps', 'Stride Number']
        if not all(col in invis_df.columns for col in required_cols):
            return metrics
        
        try:
            # Step 1: Identify success clamp periods (EXCLUDING baseline block)
            max_target = invis_df['Target size'].max()
            
            # First, find the baseline block (early in trial, const=2, max target)
            baseline_mask = (
                (invis_df['Target size'] >= max_target - 0.01) & 
                (np.abs(invis_df['Constant'] - 2.0) < 0.01) &
                (invis_df['Stride Number'] <= 50)  # Baseline should be early
            )
            
            baseline_end = 0
            if baseline_mask.any():
                baseline_strides = invis_df[baseline_mask]['Stride Number']
                baseline_end = baseline_strides.max()
                print(f"    Found baseline block ending at stride {baseline_end}")
            
            # Now find success clamps AFTER the baseline block
            clamp_mask = (
                (invis_df['Target size'] >= max_target - 0.01) & 
                (np.abs(invis_df['Constant'] - 2.0) < 0.01) &
                (invis_df['Stride Number'] > baseline_end + 10)  # Must be well after baseline
            )
            
            clamp_data = invis_df[clamp_mask].copy()
            if clamp_data.empty:
                print(f"    No success clamps found after baseline for {subject_id}")
                return metrics
                
            # Step 2: Find distinct clamp periods (AFTER baseline)
            clamp_data = clamp_data.sort_values('Stride Number')
            
            # Group consecutive strides into periods
            stride_gaps = clamp_data['Stride Number'].diff() > 1
            stride_gaps.iloc[0] = True
            clamp_data['period_id'] = stride_gaps.cumsum()
            
            # Get period information for TRUE success clamps only
            periods = []
            for period_id in clamp_data['period_id'].unique():
                period_strides = clamp_data[clamp_data['period_id'] == period_id]
                if len(period_strides) >= 15:  # Success clamps should be substantial
                    periods.append({
                        'start': period_strides['Stride Number'].min(),
                        'end': period_strides['Stride Number'].max(),
                        'data': period_strides,
                        'length': len(period_strides)
                    })
            
            # Sort by start stride
            periods.sort(key=lambda x: x['start'])
            
            print(f"    Found {len(periods)} success clamp periods after baseline")
            for i, period in enumerate(periods):
                print(f"      Clamp {i+1}: Strides {period['start']}-{period['end']} (n={period['length']})")
            
            if len(periods) < 1:
                print(f"    No valid success clamp periods for {subject_id}")
                return metrics
            
            # Step 3: Calculate retention for each TRUE success clamp period
            baseline_stride = 2.0  # Normalized baseline
    
            for i, period in enumerate(periods[:2]):  # Only first 2 TRUE clamps
                # Get clamp performance (last 20 strides of clamp)
                clamp_strides = period['data'].tail(20)['Sum of gains and steps'].mean()
                
                # Get pre-clamp performance (should be plenty of learning data before true clamps)
                pre_clamp_performance = self._get_pre_clamp_performance(
                    invis_df, period['start'], max_target
                )
                
                print(f"      Clamp {i+1}: clamp_performance={clamp_strides:.3f}, pre_clamp={pre_clamp_performance}")
                
                if pre_clamp_performance is not None:
                    # Calculate retention as percentage
                    learning_before = pre_clamp_performance - baseline_stride
                    learning_during = clamp_strides - baseline_stride
                    
                    print(f"        Learning before: {learning_before:.3f}, Learning during: {learning_during:.3f}")
                    
                    if abs(learning_before) > 0.01:  # Avoid division by very small numbers
                        retention_percent = (learning_during / learning_before) * 100
                        
                        # Store in appropriate metric
                        if i == 0:
                            metrics['invis_retention_clamp1_percent'] = retention_percent
                            print(f"  Subject {subject_id} Clamp1: {retention_percent:.1f}% retention")
                        elif i == 1:
                            metrics['invis_retention_clamp2_percent'] = retention_percent
                            print(f"  Subject {subject_id} Clamp2: {retention_percent:.1f}% retention")
                    else:
                        print(f"        Insufficient learning before clamp {i+1} (learning_before={learning_before:.3f})")
                else:
                    print(f"        No pre-clamp performance data found for clamp {i+1}")
            
            # Calculate overall retention
            clamp1 = metrics['invis_retention_clamp1_percent']
            clamp2 = metrics['invis_retention_clamp2_percent']
            
            if not pd.isna(clamp1) and not pd.isna(clamp2):
                metrics['invis_retention_overall_percent'] = (clamp1 + clamp2) / 2
            elif not pd.isna(clamp1):
                metrics['invis_retention_overall_percent'] = clamp1
            elif not pd.isna(clamp2):
                metrics['invis_retention_overall_percent'] = clamp2
            
        except Exception as e:
            print(f"  Warning: Retention calculation failed for {subject_id}: {e}")
        
        return metrics
    
    def _get_pre_clamp_performance(self, df: pd.DataFrame, clamp_start: int, max_target: float) -> Optional[float]:
        """
        Get the learning performance just before a clamp period starts.
        Since we now exclude baseline, this should work well for both clamps.
        """
        
        # Look backwards from clamp start
        search_start = max(1, clamp_start - 50)  # Look back up to 50 strides
        search_end = clamp_start - 1
        
        # Find learning data (not at max target size)
        learning_mask = (
            (df['Stride Number'] >= search_start) & 
            (df['Stride Number'] <= search_end) &
            (df['Target size'] < max_target - 0.01)  # Not max target = learning phase
        )
        
        learning_data = df[learning_mask]
        
        print(f"        Pre-clamp search: strides {search_start}-{search_end}, found {len(learning_data)} learning strides")
        
        if len(learning_data) >= 10:  # Need at least 10 strides
            # Take the last 15 strides (or all if fewer than 15)
            recent_learning = learning_data.tail(15)
            performance = recent_learning['Sum of gains and steps'].mean()
            print(f"        Using last {len(recent_learning)} learning strides, performance: {performance:.3f}")
            return performance
        
        print(f"        Insufficient pre-clamp learning data ({len(learning_data)} strides)")
        return None 

    
    def _calculate_subject_metrics(self, subject: SubjectData) -> Optional[Dict]:
        """Calculate metrics for a single subject - ENHANCED WITH RETENTION."""
        result = {
            'ID': subject.subject_id,
            'age': subject.age,
            'session_date': subject.metadata.get('Session Date'),
            'min_const_tape_score': subject.metadata.get('min_const_tape_score'),
            'pref_const_tape_score': subject.metadata.get('pref_const_tape_score'), 
            'max_const_tape_score': subject.metadata.get('max_const_tape_score')
        }
        
        # Process each trial type
        for trial_type in ['vis1', 'invis', 'vis2']:
            trial_dict = subject.trial_data.get(trial_type)
            if not trial_dict:
                continue
            
            df = trial_dict['data']
            if df is None or df.empty or 'Success' not in df.columns:
                continue
            
            # Calculate standard metrics for both conditions
            for condition in ['max', 'min']:
                period_data, indices = self._get_period_data(df, condition)
                if period_data is not None and not period_data.empty:
                    metrics = self.metrics_engine.calculate_period_metrics(
                        period_data, trial_type, condition
                    )
                    result.update(metrics)
                    result[f'{trial_type}_{condition}_const_indices'] = indices

            stride_change_metrics = self.metrics_engine.calculate_stride_change_distributions(trial_dict, trial_type)
            result.update(stride_change_metrics)
            # Add trial metadata
            if df is not None:
                result.update({
                    f'{trial_type}_min_target_size': df['Target size'].min() if 'Target size' in df.columns else None,
                    f'{trial_type}_max_constant': df['Constant'].max() if 'Constant' in df.columns else None,
                    f'{trial_type}_min_constant': df['Constant'].min() if 'Constant' in df.columns else None
                })
                
                # Order information for invis trials
                if trial_type == 'invis':
                    result.update(self._calculate_condition_order(df))
        
        # Process preference trial (existing code remains the same)
        pref_dict = subject.trial_data.get('pref')
        if pref_dict:
            pref_metrics = self.metrics_engine.calculate_preference_metrics(pref_dict['data'])
            result.update(pref_metrics)
        
        return result
    
    def _get_period_data(self, df: pd.DataFrame, condition: str, 
                        length: int = 20) -> Tuple[Optional[pd.DataFrame], Optional[List]]:
        """Extract data for specific condition period."""
        if df is None or df.empty:
            return None, None
        
        if 'Target size' not in df.columns or 'Constant' not in df.columns:
            return None, None
        
        min_target = df['Target size'].min()
        target_tolerance = 0.001
        
        min_target_periods = df[df['Target size'] <= min_target + target_tolerance]
        if min_target_periods.empty:
            return None, None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max'
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[
            np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)
        ]
        
        if period_data.empty:
            return None, None
        
        return period_data.tail(length), period_data.index.tolist()
    
    def _calculate_condition_order(self, df: pd.DataFrame) -> Dict:
        """Determine which condition came first for invis trials."""
        all_max_indices = df.index[df['Constant'] == df['Constant'].max()].tolist()
        all_min_indices = df.index[df['Constant'] == df['Constant'].min()].tolist()
        
        if all_max_indices and all_min_indices:
            first_max = min(all_max_indices)
            first_min = min(all_min_indices)
            return {
                'invis_max_first': first_max < first_min,
                'invis_min_first': first_min < first_max
            }
        
        return {'invis_max_first': False, 'invis_min_first': False}
    
    def filter_subjects(self, max_target_size: float = None, 
                       min_age: float = None, max_age: float = None,
                       required_trial_types: List[str] = None) -> 'DataManager':
        """Create a filtered DataManager instance."""
        filtered_subjects = {}
        
        for subject_id, subject in self.subjects.items():
            # Age filtering
            age = subject.age
            if min_age is not None and age < min_age:
                continue
            if max_age is not None and age > max_age:
                continue
            
            # Check required trial types
            if required_trial_types:
                missing_trials = [
                    t for t in required_trial_types 
                    if t not in subject.trial_data or 
                       subject.trial_data[t]['data'] is None
                ]
                if missing_trials:
                    continue
            
            # Apply other filters
            valid = True
            for trial_type, trial_dict in subject.trial_data.items():
                if trial_dict and trial_dict['data'] is not None:
                    df = trial_dict['data']
                    
                    if (max_target_size is not None and 
                        'Target size' in df.columns and 
                        df['Target size'].min() > max_target_size):
                        valid = False
                        break
            
            if valid:
                filtered_subjects[subject_id] = subject
        
        # Create new instance with filtered data
        new_manager = DataManager.__new__(DataManager)
        new_manager.config = self.config
        new_manager.metadata_path = self.metadata_path
        new_manager.data_root_dir = self.data_root_dir
        new_manager.debug = self.debug
        new_manager.trial_processor = self.trial_processor
        new_manager.metrics_engine = self.metrics_engine
        new_manager.subjects = filtered_subjects
        new_manager.metadata = pd.DataFrame.from_dict(
            {subj: data.metadata for subj, data in filtered_subjects.items()}, 
            orient='index'
        )
        
        return new_manager

# ==============================================================================
# STATISTICAL ANALYSIS
# ==============================================================================

class StatisticalAnalyzer:
    """Handles all statistical analyses."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        self.config = config
        self.metrics_df = metrics_df
        
        # Apply motor noise filter
        if 'mot_noise' in metrics_df.columns:
            self.filtered_df = metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_df = metrics_df

    def run_repeated_measures_anova_matched_tape_scores(self) -> Dict:
        """Run ANOVA with condition-matched tape scores (min tape score for min condition, max for max)."""
        
        # Prepare long-format data
        long_rows = []
        
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            for col in self.filtered_df.columns:
                if '_sr_' in col and '_const' in col:
                    parts = col.split('_')
                    if len(parts) >= 3:
                        trial_type = parts[0]
                        condition = parts[2]
                        
                        if pd.notna(row[col]):
                            # MATCH TAPE SCORE TO CONDITION
                            if condition == 'min':
                                matched_tape_score = row.get('min_const_tape_score', np.nan)
                            elif condition == 'max':
                                matched_tape_score = row.get('max_const_tape_score', np.nan)
                            else:
                                matched_tape_score = np.nan
                            
                            long_rows.append({
                                'subject': subject_id,
                                'trial_type': trial_type,
                                'condition': condition,
                                'success_rate': row[col],
                                'age': row.get('age', np.nan),
                                'motor_noise': row.get('mot_noise', np.nan),
                                'pref_asymmetry': row.get('pref_asymmetry', np.nan),
                                'pref_const_tape_score': row.get('pref_const_tape_score', np.nan),
                                # KEY: Condition-matched tape score
                                'tape_score_matched': matched_tape_score,
                                # Keep individual scores for comparison
                                'min_const_tape_score': row.get('min_const_tape_score', np.nan),
                                'max_const_tape_score': row.get('max_const_tape_score', np.nan)
                            })
        
        df_long = pd.DataFrame(long_rows).dropna(subset=['success_rate'])
        
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        results = {'analysis_type': 'condition_matched_tape_scores'}
        
        # Basic ANOVA
        try:
            # Main effects and interaction (same as before)
            if len(df_long['trial_type'].unique()) > 1:
                aov_trial = pg.rm_anova(data=df_long, dv='success_rate', 
                                       within='trial_type', subject='subject', detailed=True)
                results['trial_type_effect'] = {
                    'F': float(aov_trial['F'].iloc[0]),
                    'p_value': float(aov_trial['p-unc'].iloc[0]),
                    'effect_size': float(aov_trial['ng2'].iloc[0])
                }
            
            if len(df_long['condition'].unique()) > 1:
                aov_condition = pg.rm_anova(data=df_long, dv='success_rate', 
                                           within='condition', subject='subject', detailed=True)
                results['condition_effect'] = {
                    'F': float(aov_condition['F'].iloc[0]),
                    'p_value': float(aov_condition['p-unc'].iloc[0]),
                    'effect_size': float(aov_condition['ng2'].iloc[0])
                }
            
            if len(df_long['trial_type'].unique()) > 1 and len(df_long['condition'].unique()) > 1:
                aov_interaction = pg.rm_anova(data=df_long, dv='success_rate', 
                                             within=['trial_type', 'condition'], 
                                             subject='subject', detailed=True)
                interaction_row = aov_interaction[aov_interaction['Source'].str.contains('trial_type \\* condition')]
                if not interaction_row.empty:
                    results['interaction_effect'] = {
                        'F': float(interaction_row['F'].iloc[0]),
                        'p_value': float(interaction_row['p-unc'].iloc[0]),
                        'effect_size': float(interaction_row['ng2'].iloc[0])
                    }
        except Exception as e:
            results['basic_anova_error'] = str(e)
        
        # IMPROVED COVARIATE ANALYSIS WITH MATCHED TAPE SCORES
        try:
            # Standard covariates
            standard_covariates = []
            if 'age' in df_long.columns and df_long['age'].notna().sum() > 0:
                standard_covariates.append('age')
            if 'motor_noise' in df_long.columns and df_long['motor_noise'].notna().sum() > 0:
                standard_covariates.append('motor_noise')
            if 'pref_asymmetry' in df_long.columns and df_long['pref_asymmetry'].notna().sum() > 0:
                standard_covariates.append('pref_asymmetry')
            if 'pref_const_tape_score' in df_long.columns and df_long['pref_const_tape_score'].notna().sum() > 0:
                standard_covariates.append('pref_const_tape_score')
            
            # Matched tape score (key innovation)
            matched_available = 'tape_score_matched' in df_long.columns and df_long['tape_score_matched'].notna().sum() > 0
            
            results['covariates_used'] = {
                'standard': standard_covariates,
                'matched_tape_score': matched_available
            }
            
            if standard_covariates or matched_available:
                # Test matched tape score correlation
                if matched_available:
                    tape_corr = df_long.groupby('subject').agg({
                        'success_rate': 'mean',
                        'tape_score_matched': 'mean'
                    }).reset_index()
                    
                    if len(tape_corr) > 5:
                        from scipy.stats import pearsonr
                        r, p = pearsonr(tape_corr['tape_score_matched'], tape_corr['success_rate'])
                        results['matched_tape_score_effect'] = {
                            'correlation': float(r),
                            'p_value': float(p),
                            'interpretation': 'Condition-matched tape score effect on success rate',
                            'n_subjects': len(tape_corr)
                        }
                
                # Mixed effects models - THREE DIFFERENT APPROACHES
                try:
                    import statsmodels.api as sm
                    from statsmodels.formula.api import mixedlm
                    
                    # MODEL 1: Simple additive matched tape score
                    if matched_available:
                        all_covs_simple = standard_covariates + ['tape_score_matched']
                        formula_simple = f"success_rate ~ trial_type * condition + {' + '.join(all_covs_simple)}"
                        
                        model_simple = mixedlm(formula_simple, df_long, groups=df_long['subject'])
                        fitted_simple = model_simple.fit()
                        
                        results['model_1_simple_matched'] = {
                            'formula': formula_simple,
                            'aic': float(fitted_simple.aic),
                            'bic': float(fitted_simple.bic)
                        }
                        
                        if 'tape_score_matched' in fitted_simple.params.index:
                            results['model_1_tape_effect'] = {
                                'coefficient': float(fitted_simple.params['tape_score_matched']),
                                'p_value': float(fitted_simple.pvalues['tape_score_matched']),
                                'confidence_interval': [float(fitted_simple.conf_int().loc['tape_score_matched', 0]), 
                                                      float(fitted_simple.conf_int().loc['tape_score_matched', 1])],
                                'interpretation': 'Change in success rate per unit increase in condition-matched tape score'
                            }
                    
                    # MODEL 2: Test if tape score effect differs by condition
                    if matched_available:
                        formula_interaction = f"success_rate ~ trial_type * condition * tape_score_matched + {' + '.join(standard_covariates)}"
                        
                        try:
                            model_interaction = mixedlm(formula_interaction, df_long, groups=df_long['subject'])
                            fitted_interaction = model_interaction.fit()
                            
                            results['model_2_tape_condition_interaction'] = {
                                'formula': formula_interaction,
                                'aic': float(fitted_interaction.aic),
                                'bic': float(fitted_interaction.bic)
                            }
                            
                            # Test if tape score effect differs by condition
                            interaction_terms = [param for param in fitted_interaction.params.index 
                                               if 'tape_score_matched' in param and ':' in param]
                            
                            if interaction_terms:
                                results['model_2_interactions'] = {}
                                for term in interaction_terms:
                                    results['model_2_interactions'][term] = {
                                        'coefficient': float(fitted_interaction.params[term]),
                                        'p_value': float(fitted_interaction.pvalues[term]),
                                        'interpretation': f'How tape score effect differs for {term}'
                                    }
                        
                        except Exception as e:
                            results['model_2_error'] = str(e)
                    
                    # MODEL 3: Compare condition-specific vs general tape scores
                    if all(score in df_long.columns for score in ['min_const_tape_score', 'max_const_tape_score']):
                        # Model with separate min/max tape scores (original approach)
                        formula_separate = f"success_rate ~ trial_type * condition + min_const_tape_score + max_const_tape_score + {' + '.join(standard_covariates)}"
                        
                        try:
                            model_separate = mixedlm(formula_separate, df_long, groups=df_long['subject'])
                            fitted_separate = model_separate.fit()
                            
                            results['model_3_separate_scores'] = {
                                'formula': formula_separate,
                                'aic': float(fitted_separate.aic),
                                'bic': float(fitted_separate.bic)
                            }
                            
                            # Compare AIC/BIC between matched vs separate models
                            if 'model_1_simple_matched' in results:
                                aic_matched = results['model_1_simple_matched']['aic']
                                aic_separate = results['model_3_separate_scores']['aic']
                                
                                results['model_comparison'] = {
                                    'matched_tape_aic': aic_matched,
                                    'separate_tape_aic': aic_separate,
                                    'matched_is_better': aic_matched < aic_separate,
                                    'aic_difference': aic_separate - aic_matched,
                                    'interpretation': 'Negative difference means matched model is better'
                                }
                        
                        except Exception as e:
                            results['model_3_error'] = str(e)
                
                except Exception as e:
                    results['mixed_effects_error'] = str(e)
            
        except Exception as e:
            results['covariate_analysis_error'] = str(e)
        
        # Descriptive statistics by condition
        results['descriptive_stats'] = {
            'n_subjects': len(df_long['subject'].unique()),
            'n_observations': len(df_long),
            'tape_score_by_condition': {
                'min_condition': {
                    'mean_tape_score': float(df_long[df_long['condition'] == 'min']['tape_score_matched'].mean()),
                    'std_tape_score': float(df_long[df_long['condition'] == 'min']['tape_score_matched'].std()),
                    'n_valid': int(df_long[df_long['condition'] == 'min']['tape_score_matched'].notna().sum())
                },
                'max_condition': {
                    'mean_tape_score': float(df_long[df_long['condition'] == 'max']['tape_score_matched'].mean()),
                    'std_tape_score': float(df_long[df_long['condition'] == 'max']['tape_score_matched'].std()),
                    'n_valid': int(df_long[df_long['condition'] == 'max']['tape_score_matched'].notna().sum())
                }
            }
        }
        
        return results


    def _get_covariates_with_tape_scores(self, df_long):
        """Get all available covariates including tape scores."""
        covariates = []
        
        # Standard covariates
        standard_covs = ['age', 'motor_noise', 'pref_asymmetry']
        # Tape score covariates  
        tape_covs = ['min_const_tape_score', 'pref_const_tape_score', 'max_const_tape_score']
        
        for cov in standard_covs + tape_covs:
            if cov in df_long.columns and df_long[cov].notna().sum() > 0:
                covariates.append(cov)
        
        return covariates
    
    
    def _interpret_cohens_d(self, d):
        """Interpret Cohen's d effect size (add this helper method)."""
        abs_d = abs(d)
        if abs_d < 0.2:
            return "negligible"
        elif abs_d < 0.5:
            return "small"
        elif abs_d < 0.8:
            return "medium"
        else:
            return "large"
    
    def create_age_comparison_report(age_results: Dict) -> str:
        """
        Generate a text report summarizing age-stratified ANOVA results.
        """
        
        if 'group_analyses' not in age_results:
            return "No group analyses found in results."
        
        report = []
        report.append("=" * 80)
        report.append("AGE-STRATIFIED ANOVA ANALYSIS REPORT")
        report.append("=" * 80)
        report.append("")
        
        # Overview
        total_subjects = age_results.get('total_subjects_analyzed', 'unknown')
        age_range = age_results.get('age_range', ['unknown', 'unknown'])
        report.append(f"Total subjects analyzed: {total_subjects}")
        report.append(f"Age range: {age_range[0]:.1f} - {age_range[1]:.1f} years")
        report.append("")
        
        # Age group definitions
        if 'age_group_definitions' in age_results:
            report.append("AGE GROUP DEFINITIONS:")
            for group, (min_age, max_age) in age_results['age_group_definitions'].items():
                report.append(f"  {group.title()}: {min_age} - {max_age} years")
            report.append("")
        
        # Group assignments
        if 'group_assignments' in age_results:
            report.append("ACTUAL GROUP ASSIGNMENTS:")
            for group, info in age_results['group_assignments'].items():
                if info['n_subjects'] > 0:
                    report.append(f"  {group.title()}: n={info['n_subjects']}, "
                                f"age range={info['age_range'][0]:.1f}-{info['age_range'][1]:.1f}, "
                                f"mean age={info['mean_age']:.1f}")
                else:
                    report.append(f"  {group.title()}: No subjects")
            report.append("")
        
        # Individual group results
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results]
        
        if valid_groups:
            report.append("INDIVIDUAL GROUP RESULTS:")
            report.append("-" * 50)
            
            for group in valid_groups:
                group_data = age_results['group_analyses'][group]
                report.append(f"\n{group.upper()} GROUP:")
                
                # Sample info
                if 'descriptive_stats' in group_data:
                    n_subj = group_data['descriptive_stats']['n_subjects']
                    mean_age = group_data['descriptive_stats']['age_info']['mean_age']
                    report.append(f"  Sample: n={n_subj}, mean age={mean_age:.1f} years")
                
                # Trial type effect
                if 'trial_type_effect' in group_data:
                    effect = group_data['trial_type_effect']
                    report.append(f"  Trial Type Effect: F={effect['F']:.2f}, p={effect['p_value']:.3f}, η²={effect['effect_size_eta2']:.3f}")
                    report.append(f"  Significant: {'YES' if effect['significant'] else 'NO'}")
                    
                    # Effect size interpretation
                    eta2 = effect['effect_size_eta2']
                    if eta2 < 0.01:
                        interp = "negligible"
                    elif eta2 < 0.06:
                        interp = "small"
                    elif eta2 < 0.14:
                        interp = "medium"
                    else:
                        interp = "large"
                    report.append(f"  Effect size interpretation: {interp}")
                
                # Mean success rates
                if 'mean_success_rates' in group_data:
                    report.append("  Mean success rates:")
                    for trial, rate in group_data['mean_success_rates'].items():
                        report.append(f"    {trial}: {rate:.3f}")
                
                # Post-hoc comparisons if available
                if 'posthoc_comparisons' in group_data:
                    report.append("  Post-hoc comparisons:")
                    for comparison, results in group_data['posthoc_comparisons'].items():
                        sig_text = "***" if results['significant'] else "ns"
                        report.append(f"    {comparison}: d={results['cohens_d']:.3f} ({results['effect_size_interpretation']}), "
                                    f"p={results['p_corrected']:.3f} {sig_text}")
        
        # Between-group comparisons
        if 'between_group_comparisons' in age_results and 'error' not in age_results['between_group_comparisons']:
            report.append("\n" + "=" * 50)
            report.append("BETWEEN-GROUP COMPARISONS:")
            report.append("=" * 50)
            
            between_data = age_results['between_group_comparisons']
            if 'group_comparison_summary' in between_data:
                comparisons_made = between_data['group_comparison_summary'].get('comparisons_made', [])
                
                for comparison in comparisons_made:
                    if comparison in between_data:
                        report.append(f"\n{comparison.upper()}:")
                        comp_data = between_data[comparison]
                        
                        for trial, results in comp_data.items():
                            sig_text = "***" if results['significant'] else "ns"
                            report.append(f"  {trial}: {results['group1_mean']:.3f} vs {results['group2_mean']:.3f}, "
                                        f"diff={results['mean_difference']:.3f}, p={results['p_value']:.3f} {sig_text}")
        
        # Summary interpretation
        if 'summary_comparison' in age_results and 'error' not in age_results['summary_comparison']:
            report.append("\n" + "=" * 50)
            report.append("SUMMARY & INTERPRETATION:")
            report.append("=" * 50)
            
            summary = age_results['summary_comparison']
            
            if 'pattern_consistency' in summary:
                consistency = summary['pattern_consistency']
                report.append(f"\nPattern Consistency:")
                report.append(f"  Total age groups analyzed: {consistency['total_groups']}")
                report.append(f"  Groups with significant trial effect: {consistency['groups_with_significant_effect']}")
                report.append(f"  Proportion with significant effect: {consistency['proportion_significant']:.1%}")
                report.append(f"  Consistent across all groups: {'YES' if consistency['consistent_effect'] else 'NO'}")
                
                if consistency['groups_with_significant_effect_names']:
                    report.append(f"  Groups showing significant effects: {', '.join(consistency['groups_with_significant_effect_names'])}")
            
            # Interpretation recommendations
            report.append(f"\nInterpretation:")
            if 'pattern_consistency' in summary:
                prop_sig = summary['pattern_consistency']['proportion_significant']
                if prop_sig >= 0.8:
                    report.append("  - Strong evidence for trial type effects across age groups")
                    report.append("  - Visual feedback appears important for motor learning at all ages")
                elif prop_sig >= 0.5:
                    report.append("  - Mixed evidence for trial type effects across age groups")
                    report.append("  - Some age groups may be more sensitive to visual feedback than others")
                else:
                    report.append("  - Weak evidence for consistent trial type effects")
                    report.append("  - Age may moderate the importance of visual feedback")
            
            # Effect size patterns
            if 'effect_size_comparison' in summary:
                effect_sizes = [(group, data['eta_squared']) for group, data in summary['effect_size_comparison'].items()]
                effect_sizes.sort(key=lambda x: x[1], reverse=True)
                
                report.append(f"\nEffect Size Pattern (largest to smallest):")
                for group, eta2 in effect_sizes:
                    interpretation = summary['effect_size_comparison'][group]['interpretation']
                    report.append(f"  {group}: η²={eta2:.3f} ({interpretation})")
        
        report.append("\n" + "=" * 80)
        report.append("END REPORT")
        report.append("=" * 80)
        
        return "\n".join(report)
    
    # Usage examples for the complete age analysis:
    
    """
    # Example 1: Standard age analysis
    age_results = stats.run_age_stratified_anova()
    
    # Generate report
    report = create_age_comparison_report(age_results)
    print(report)
    
    # Create visualizations
    fig1 = pop_viz.plot_age_stratified_results(age_results, 'age_anova_comparison.png')
    fig2 = pop_viz.plot_trial_patterns_by_age(age_results, 'trial_patterns_by_age.png')
    
    # Example 2: Custom age groups focusing on specific developmental periods
    custom_groups = {
        'pre_adolescent': [7, 11],
        'early_adolescent': [11, 14], 
        'mid_adolescent': [14, 17],
        'late_adolescent': [17, 18]
    }
    
    custom_results = stats.run_age_stratified_anova(
        age_groups=custom_groups,
        min_subjects_per_group=12
    )
    
    # Example 3: Binary split at median age
    import numpy as np
    median_age = stats.filtered_df['age'].median()
    binary_groups = {
        'younger_half': [7, median_age],
        'older_half': [median_age, 18]
    }
    
    binary_results = stats.run_age_stratified_anova(age_groups=binary_groups)
    
    # Example 4: Multiple schemes comparison
    all_schemes_results = stats.run_custom_age_analyses()
    
    # Compare results across schemes
    for scheme_name, results in all_schemes_results.items():
        if 'summary_comparison' in results:
            consistency = results['summary_comparison'].get('pattern_consistency', {})
            print(f"{scheme_name}: {consistency.get('proportion_significant', 0):.1%} of groups significant")
    """
    
    def run_regression_analysis(self, trial_type: str = 'invis', 
                               condition: str = 'max',
                               predictors: List[str] = None) -> Dict:
        """Run regression analysis."""
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
        
        available_predictors = [p for p in predictors if p in self.filtered_df.columns]
        target_col = f'{trial_type}_sr_{condition}_const'
        
        if target_col not in self.filtered_df.columns:
            raise ValueError(f"Target column {target_col} not found")
        
        # Prepare data
        valid_data = self.filtered_df[available_predictors + [target_col]].dropna()
        
        if len(valid_data) < 10:
            raise ValueError(f"Insufficient data: only {len(valid_data)} valid samples")
        
        X = valid_data[available_predictors]
        y = valid_data[target_col]
        
        # Split and train
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', LinearRegression())
        ])
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        return {
            'model': model,
            'metrics': {
                'r2': r2_score(y_test, y_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
                'n_samples': len(valid_data)
            },
            'feature_importances': dict(zip(available_predictors, 
                                          np.abs(model.named_steps['regressor'].coef_)))
        }

    def run_unified_age_stratified_anova(self, 
                                         dependent_var_pattern: str,
                                         dependent_var_name: str = None,
                                         predictors: List[str] = None,
                                         age_groups: Dict[str, List[float]] = None, 
                                         min_subjects_per_group: int = 1) -> Dict:
        """
        Unified age-stratified ANOVA for any dependent variable pattern.
        FIXED: Proper handling of retention data with correct effect testing.
        """
        
        # Defaults
        if age_groups is None:
            age_groups = {'younger': [7, 12], 'middle': [12, 15], 'older': [15, 18]}
        
        if dependent_var_name is None:
            names = {'_sr_': 'Success Rate', '_learning_': 'Learning', '_retention_': 'Retention', 
                    '_msl_': 'Mean Stride Length', '_sd_': 'Stride Variability'}
            dependent_var_name = names.get(dependent_var_pattern, 'Dependent Variable')
        
        if predictors is None:
            predictors = ['age', 'mot_noise', 'pref_asymmetry']
            # Add available tape scores
            for tape in ['min_const_tape_score', 'pref_const_tape_score', 'max_const_tape_score']:
                if tape in self.filtered_df.columns:
                    predictors.append(tape)
        
        # FIXED: Find dependent variable columns with updated logic for retention
        if dependent_var_pattern == '_retention_':
            # Look for the new retention column names with 'percent'
            dependent_cols = [col for col in self.filtered_df.columns 
                             if 'retention' in col and 'percent' in col and 'invis' in col]
            print(f"Found retention columns: {dependent_cols}")
        else:
            # Original logic for other patterns
            dependent_cols = [col for col in self.filtered_df.columns 
                             if dependent_var_pattern in col and '_const' in col]
        
        if not dependent_cols:
            return {'error': f'No columns found matching pattern: {dependent_var_pattern}'}
        
        # Prepare long-format data
        long_rows = []
        for _, row in self.filtered_df.iterrows():
            subject_id = row.name
            age = row.get('age', np.nan)
            
            if pd.isna(age):
                continue
            
            for col in dependent_cols:
                if pd.notna(row[col]):
                    if dependent_var_pattern == '_retention_':
                        # FIXED: Handle retention columns specially
                        if 'clamp1' in col:
                            condition_type, trial_type = 'clamp1', 'invis'
                        elif 'clamp2' in col:
                            condition_type, trial_type = 'clamp2', 'invis'
                        elif 'overall' in col:
                            condition_type, trial_type = 'overall', 'invis'
                        else:
                            continue
                        
                        row_data = {
                            'subject': subject_id, 'clamp_type': condition_type, 'trial_type': trial_type,
                            'retention': row[col], 'age': age  # Use 'retention' as the DV name
                        }
                    else:
                        # Standard parsing for other metrics
                        parts = col.split('_')
                        if len(parts) >= 3:
                            trial_type = parts[0]
                            condition = parts[2] if len(parts) > 2 else 'unknown'
                            row_data = {
                                'subject': subject_id, 'trial_type': trial_type, 'condition': condition,
                                dependent_var_pattern.strip('_'): row[col], 'age': age
                            }
                        else:
                            continue
                    
                    # Add predictors
                    for pred in predictors:
                        row_data[pred] = row.get(pred, np.nan)
                    
                    long_rows.append(row_data)
        
        df_long = pd.DataFrame(long_rows)
        
        # Set the correct dependent variable column name
        dependent_var_col = 'retention' if dependent_var_pattern == '_retention_' else dependent_var_pattern.strip('_')
        df_long = df_long.dropna(subset=[dependent_var_col, 'age'])
        
        if df_long.empty:
            return {'error': 'No valid data for analysis'}
        
        print(f"Created long-format data with {len(df_long)} observations for {dependent_var_name}")
        print(f"Dependent variable column: {dependent_var_col}")
        print(f"Unique values in dependent variable: {df_long[dependent_var_col].describe()}")
        
        # Initialize results
        results = {
            'dependent_variable': dependent_var_name,
            'age_group_definitions': age_groups,
            'total_subjects_analyzed': len(df_long['subject'].unique()),
            'predictors_used': [p for p in predictors if p in df_long.columns and df_long[p].notna().sum() > 0],
            'group_analyses': {},
            'summary_comparison': {}
        }
        
        # Assign age groups
        df_long['age_group'] = None
        group_assignments = {}
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_long['age'] >= min_age) & (df_long['age'] < max_age)
            df_long.loc[mask, 'age_group'] = group_name
            
            subjects_in_group = df_long[mask]['subject'].unique()
            group_assignments[group_name] = {
                'n_subjects': len(subjects_in_group),
                'mean_age': float(df_long[mask]['age'].mean()) if len(subjects_in_group) > 0 else np.nan
            }
        
        results['group_assignments'] = group_assignments
        
        # Run ANOVA for each age group
        for group_name, group_info in group_assignments.items():
            if group_info['n_subjects'] < min_subjects_per_group:
                results['group_analyses'][group_name] = {
                    'error': f'Insufficient subjects (n={group_info["n_subjects"]})'
                }
                continue
            
            group_data = df_long[df_long['age_group'] == group_name].copy()
            group_results = {'descriptive_stats': {'n_subjects': group_info['n_subjects']}}
            
            try:
                import pingouin as pg
                
                # FIXED: Proper handling based on data type
                if dependent_var_pattern == '_retention_':
                    # For retention: Test clamp type effect only (clamp1 vs clamp2)
                    anova_data = group_data[group_data['clamp_type'] != 'overall']
                    
                    if len(anova_data['clamp_type'].unique()) > 1:
                        aov = pg.rm_anova(data=anova_data, dv=dependent_var_col, 
                                         within='clamp_type', subject='subject', detailed=True)
                        
                        group_results['clamp_type_effect'] = {
                            'F': float(aov['F'].iloc[0]),
                            'p_value': float(aov['p-unc'].iloc[0]),
                            'effect_size_eta2': float(aov['ng2'].iloc[0]),
                            'significant': float(aov['p-unc'].iloc[0]) < 0.05,
                            'interpretation': 'Difference between clamp1 and clamp2 retention'
                        }
                        
                        # FIXED: For visualizations that expect 'trial_type_effect', 
                        # use clamp_type_effect but clarify what it represents
                        group_results['trial_type_effect'] = group_results['clamp_type_effect'].copy()
                        group_results['trial_type_effect']['interpretation'] = 'Clamp type effect (clamp1 vs clamp2) - shown as trial effect for visualization compatibility'
                        
                        # Calculate mean retention by clamp type for descriptive stats
                        group_results['mean_retention_by_clamp'] = {}
                        for clamp_type in anova_data['clamp_type'].unique():
                            clamp_data = anova_data[anova_data['clamp_type'] == clamp_type][dependent_var_col]
                            group_results['mean_retention_by_clamp'][clamp_type] = float(clamp_data.mean())
                    
                    else:
                        group_results['clamp_type_effect'] = {
                            'error': 'Only one clamp type found - cannot test clamp effect'
                        }
                        # For visualizations - create a dummy trial_type_effect
                        group_results['trial_type_effect'] = {
                            'F': 0.0,
                            'p_value': 1.0,
                            'effect_size_eta2': 0.0,
                            'significant': False,
                            'interpretation': 'No clamp type variation - cannot test effect'
                        }
                
                else:
                    # For other measures: Test trial type effect (vis1, invis, vis2)
                    if len(group_data['trial_type'].unique()) > 1:
                        aov = pg.rm_anova(data=group_data, dv=dependent_var_col, 
                                         within='trial_type', subject='subject', detailed=True)
                        group_results['trial_type_effect'] = {
                            'F': float(aov['F'].iloc[0]),
                            'p_value': float(aov['p-unc'].iloc[0]),
                            'effect_size_eta2': float(aov['ng2'].iloc[0]),
                            'significant': float(aov['p-unc'].iloc[0]) < 0.05,
                            'interpretation': 'Difference across trial types (vis1, invis, vis2)'
                        }
                    
                    # Test condition effect if applicable (max vs min)
                    if 'condition' in group_data.columns and len(group_data['condition'].unique()) > 1:
                        aov = pg.rm_anova(data=group_data, dv=dependent_var_col, 
                                         within='condition', subject='subject', detailed=True)
                        group_results['condition_effect'] = {
                            'F': float(aov['F'].iloc[0]),
                            'p_value': float(aov['p-unc'].iloc[0]),
                            'effect_size_eta2': float(aov['ng2'].iloc[0]),
                            'significant': float(aov['p-unc'].iloc[0]) < 0.05,
                            'interpretation': 'Difference between max and min conditions'
                        }
            
            except Exception as e:
                group_results['anova_error'] = str(e)
                print(f"ANOVA error for group {group_name}: {e}")
            
            # Covariate analysis
            try:
                available_predictors = [p for p in results['predictors_used'] if p in group_data.columns]
                
                if available_predictors:
                    # Individual correlations
                    subject_means = group_data.groupby('subject').agg({
                        dependent_var_col: 'mean',
                        **{pred: 'first' for pred in available_predictors}
                    }).reset_index()
                    
                    for pred in available_predictors:
                        if subject_means[pred].notna().sum() > 5:
                            from scipy.stats import pearsonr
                            valid_data = subject_means[[pred, dependent_var_col]].dropna()
                            if len(valid_data) > 5:
                                r, p = pearsonr(valid_data[pred], valid_data[dependent_var_col])
                                group_results[f'{pred}_covariate_effect'] = {
                                    'correlation': float(r),
                                    'p_value': float(p),
                                    'interpretation': f'{pred} effect on {dependent_var_name.lower()}'
                                }
            
            except Exception as e:
                group_results['covariate_analysis_error'] = str(e)
            
            results['group_analyses'][group_name] = group_results
        
        return results
    
    
    
    def run_correlation_analysis(self) -> Dict:
        """Run correlation analysis between key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        # ADD TAPE SCORES TO CORRELATION ANALYSIS
        tape_scores = ['min_const_tape_score', 'pref_const_tape_score', 'max_const_tape_score']
        for tape_score in tape_scores:
            if tape_score in self.filtered_df.columns:
                key_cols.append(tape_score)
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols)
        
        # Calculate correlation matrix
        corr_data = self.filtered_df[key_cols].corr()
        
        # Extract significant correlations
        significant_corrs = {}
        for i, col1 in enumerate(key_cols):
            for j, col2 in enumerate(key_cols):
                if i < j:  # Avoid duplicates
                    valid_data = self.filtered_df[[col1, col2]].dropna()
                    if len(valid_data) > 5:
                        r, p = pearsonr(valid_data[col1], valid_data[col2])
                        if p < 0.05:
                            significant_corrs[f'{col1}_vs_{col2}'] = {
                                'r': r,
                                'p': p,
                                'n': len(valid_data)
                            }
        
        return {
            'correlation_matrix': corr_data.to_dict(),
            'significant_correlations': significant_corrs
        }

# ==============================================================================
# VISUALIZATION COMPONENTS
# ==============================================================================

class PopulationVisualizer(BaseVisualizer):
    """Handles population-level visualizations."""
    
    def __init__(self, metrics_df: pd.DataFrame, config: AnalysisConfig):
        super().__init__(config)
        self.metrics_df = metrics_df
        
        # Apply motor noise filter
        if 'mot_noise' in metrics_df.columns:
            self.filtered_df = metrics_df[metrics_df['mot_noise'] <= config.motor_noise_threshold]
        else:
            self.filtered_df = metrics_df

# Add these methods to the PopulationVisualizer class

    def plot_individual_stride_change_distributions(self, max_subjects: int = 20) -> Path:
        """
        Plot stride change distributions for individual participants.
        Shows success vs failure distributions for each person.
        """
        
        # Check if we have stride change data
        stride_cols = [col for col in self.filtered_df.columns if 'stride_changes_after' in col]
        if not stride_cols:
            print("No stride change data found")
            return None
        
        # Get subjects with stride change data
        subjects_with_data = []
        for _, row in self.filtered_df.iterrows():
            has_data = False
            for trial_type in ['vis1', 'invis', 'vis2']:
                success_col = f'{trial_type}_stride_changes_after_success'
                if success_col in self.filtered_df.columns:
                    success_data = row[success_col]
                    if isinstance(success_data, list) and len(success_data) > 0:
                        has_data = True
                        break
            if has_data:
                subjects_with_data.append(row)
        
        if not subjects_with_data:
            print("No subjects with stride change data found")
            return None
        
        # Limit number of subjects to plot
        subjects_to_plot = subjects_with_data[:max_subjects]
        n_subjects = len(subjects_to_plot)
        
        # Create subplot grid
        cols = 3  # vis1, invis, vis2
        rows = n_subjects
        fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
        
        if rows == 1:
            axes = axes.reshape(1, -1)
        
        for i, subject_row in enumerate(subjects_to_plot):
            subject_id = subject_row['ID']
            subject_age = subject_row.get('age', np.nan)
            
            for j, trial_type in enumerate(['vis1', 'invis', 'vis2']):
                ax = axes[i, j]
                
                success_col = f'{trial_type}_stride_changes_after_success'
                failure_col = f'{trial_type}_stride_changes_after_failure'
                
                success_changes = subject_row.get(success_col, [])
                failure_changes = subject_row.get(failure_col, [])
                
                # Ensure they're lists
                if not isinstance(success_changes, list):
                    success_changes = []
                if not isinstance(failure_changes, list):
                    failure_changes = []
                
                # Plot distributions if we have data
                if len(success_changes) > 0:
                    ax.hist(success_changes, bins=min(10, len(success_changes)), 
                           alpha=0.7, color='green', label=f'Success (n={len(success_changes)})')
                
                if len(failure_changes) > 0:
                    ax.hist(failure_changes, bins=min(10, len(failure_changes)), 
                           alpha=0.7, color='red', label=f'Failure (n={len(failure_changes)})')
                
                # Add reference line at zero
                ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
                
                # Formatting
                if i == 0:  # Top row
                    ax.set_title(f'{trial_type.upper()}')
                
                if j == 0:  # Left column
                    ax.set_ylabel(f'{subject_id}\n(age: {subject_age:.1f})\nFrequency')
                
                if i == rows - 1:  # Bottom row
                    ax.set_xlabel('Stride Change')
                
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)
                
                # Set consistent x-axis if possible
                all_changes = success_changes + failure_changes
                if len(all_changes) > 0:
                    margin = np.std(all_changes) * 0.5 if len(all_changes) > 1 else 0.1
                    ax.set_xlim(min(all_changes) - margin, max(all_changes) + margin)
        
        plt.suptitle(f'Individual Stride Change Distributions (First {n_subjects} Subjects)', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'individual_stride_change_distributions.png', 'population')
    
    def plot_age_group_stride_change_distributions(self, age_groups=None) -> Path:
        """
        Plot combined stride change distributions by age group.
        Combines all participants within each age group.
        """
        
        if age_groups is None:
            age_groups = {'younger': [7, 12], 'middle': [12, 15], 'older': [15, 18]}
        
        # Check if we have stride change data
        stride_cols = [col for col in self.filtered_df.columns if 'stride_changes_after' in col]
        if not stride_cols:
            print("No stride change data found")
            return None
        
        # Assign age groups
        df_with_groups = self.filtered_df.copy()
        df_with_groups['age_group'] = None
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_with_groups['age'] >= min_age) & (df_with_groups['age'] < max_age)
            df_with_groups.loc[mask, 'age_group'] = group_name
        
        # Remove subjects without age group assignment
        df_with_groups = df_with_groups.dropna(subset=['age_group'])
        
        # Create subplot grid: age_groups x trial_types
        age_group_names = list(age_groups.keys())
        trial_types = ['vis1', 'invis', 'vis2']
        
        fig, axes = plt.subplots(len(age_group_names), len(trial_types), figsize=(15, 5*len(age_group_names)))
        
        if len(age_group_names) == 1:
            axes = axes.reshape(1, -1)
        
        for i, group_name in enumerate(age_group_names):
            group_data = df_with_groups[df_with_groups['age_group'] == group_name]
            
            for j, trial_type in enumerate(trial_types):
                ax = axes[i, j]
                
                # Combine all success and failure changes for this age group and trial type
                all_success_changes = []
                all_failure_changes = []
                
                success_col = f'{trial_type}_stride_changes_after_success'
                failure_col = f'{trial_type}_stride_changes_after_failure'
                
                for _, row in group_data.iterrows():
                    success_changes = row.get(success_col, [])
                    failure_changes = row.get(failure_col, [])
                    
                    if isinstance(success_changes, list):
                        all_success_changes.extend(success_changes)
                    if isinstance(failure_changes, list):
                        all_failure_changes.extend(failure_changes)
                
                # Plot combined distributions
                if len(all_success_changes) > 0:
                    ax.hist(all_success_changes, bins=30, alpha=0.7, color='green', 
                           label=f'After Success (n={len(all_success_changes)})', density=True)
                
                if len(all_failure_changes) > 0:
                    ax.hist(all_failure_changes, bins=30, alpha=0.7, color='red', 
                           label=f'After Failure (n={len(all_failure_changes)})', density=True)
                
                # Add statistics
                if len(all_success_changes) > 0 and len(all_failure_changes) > 0:
                    success_mean = np.mean(all_success_changes)
                    failure_mean = np.mean(all_failure_changes)
                    
                    ax.axvline(x=success_mean, color='darkgreen', linestyle='-', linewidth=2,
                              label=f'Success Mean: {success_mean:.3f}')
                    ax.axvline(x=failure_mean, color='darkred', linestyle='-', linewidth=2,
                              label=f'Failure Mean: {failure_mean:.3f}')
                    
                    # Statistical test
                    from scipy.stats import mannwhitneyu
                    try:
                        stat, p_value = mannwhitneyu(all_success_changes, all_failure_changes, 
                                                   alternative='two-sided')
                        sig_text = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
                        ax.text(0.05, 0.95, f'p = {p_value:.3f} {sig_text}', 
                               transform=ax.transAxes, fontsize=10,
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                               verticalalignment='top')
                    except Exception:
                        pass
                
                # Add reference line at zero
                ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
                
                # Formatting
                if i == 0:  # Top row
                    ax.set_title(f'{trial_type.upper()}')
                
                if j == 0:  # Left column
                    n_subjects = len(group_data)
                    min_age, max_age = age_groups[group_name]
                    ax.set_ylabel(f'{group_name.title()}\n({min_age}-{max_age}y, n={n_subjects})\nDensity')
                
                if i == len(age_group_names) - 1:  # Bottom row
                    ax.set_xlabel('Stride Change')
                
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)
        
        plt.suptitle('Stride Change Distributions by Age Group', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_group_stride_change_distributions.png', 'population')
    
    def plot_stride_change_comparison_summary(self, age_groups=None) -> Path:
        """
        Create a summary plot comparing success vs failure effects across age groups.
        """
        
        if age_groups is None:
            age_groups = {'younger': [7, 12], 'middle': [12, 15], 'older': [15, 18]}
        
        # Assign age groups
        df_with_groups = self.filtered_df.copy()
        df_with_groups['age_group'] = None
        
        for group_name, (min_age, max_age) in age_groups.items():
            mask = (df_with_groups['age'] >= min_age) & (df_with_groups['age'] < max_age)
            df_with_groups.loc[mask, 'age_group'] = group_name
        
        df_with_groups = df_with_groups.dropna(subset=['age_group'])
        
        # Create summary statistics
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        age_group_names = list(age_groups.keys())
        trial_types = ['vis1', 'invis', 'vis2']
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
        
        for j, trial_type in enumerate(trial_types):
            # Plot 1: Mean differences (success - failure) by age group
            ax1 = axes[0, j]
            mean_diffs = []
            group_labels = []
            
            for group_name in age_group_names:
                group_data = df_with_groups[df_with_groups['age_group'] == group_name]
                
                all_success_changes = []
                all_failure_changes = []
                
                success_col = f'{trial_type}_stride_changes_after_success'
                failure_col = f'{trial_type}_stride_changes_after_failure'
                
                for _, row in group_data.iterrows():
                    success_changes = row.get(success_col, [])
                    failure_changes = row.get(failure_col, [])
                    
                    if isinstance(success_changes, list):
                        all_success_changes.extend(success_changes)
                    if isinstance(failure_changes, list):
                        all_failure_changes.extend(failure_changes)
                
                if len(all_success_changes) > 0 and len(all_failure_changes) > 0:
                    mean_diff = np.mean(all_success_changes) - np.mean(all_failure_changes)
                    mean_diffs.append(mean_diff)
                    group_labels.append(f'{group_name.title()}\\n(n={len(group_data)})')
            
            if mean_diffs:
                bars = ax1.bar(group_labels, mean_diffs, color=colors[j], alpha=0.7)
                ax1.axhline(y=0, color='black', linestyle='-', alpha=0.3)
                ax1.set_title(f'{trial_type.upper()}: Success - Failure')
                ax1.set_ylabel('Mean Difference in Stride Change')
                ax1.grid(True, alpha=0.3)
                
                # Add value labels on bars
                for bar, diff in zip(bars, mean_diffs):
                    height = bar.get_height()
                    ax1.text(bar.get_x() + bar.get_width()/2., height + (0.001 if height > 0 else -0.001),
                            f'{diff:.3f}', ha='center', va='bottom' if height > 0 else 'top', 
                            fontweight='bold')
            
            # Plot 2: Standard deviations by age group
            ax2 = axes[1, j]
            success_stds = []
            failure_stds = []
            
            for group_name in age_group_names:
                group_data = df_with_groups[df_with_groups['age_group'] == group_name]
                
                all_success_changes = []
                all_failure_changes = []
                
                for _, row in group_data.iterrows():
                    success_changes = row.get(success_col, [])
                    failure_changes = row.get(failure_col, [])
                    
                    if isinstance(success_changes, list):
                        all_success_changes.extend(success_changes)
                    if isinstance(failure_changes, list):
                        all_failure_changes.extend(failure_changes)
                
                if len(all_success_changes) > 0:
                    success_stds.append(np.std(all_success_changes))
                else:
                    success_stds.append(0)
                    
                if len(all_failure_changes) > 0:
                    failure_stds.append(np.std(all_failure_changes))
                else:
                    failure_stds.append(0)
            
            if success_stds and failure_stds:
                x_pos = np.arange(len(group_labels))
                width = 0.35
                
                ax2.bar(x_pos - width/2, success_stds, width, label='After Success', 
                       color='green', alpha=0.7)
                ax2.bar(x_pos + width/2, failure_stds, width, label='After Failure', 
                       color='red', alpha=0.7)
                
                ax2.set_title(f'{trial_type.upper()}: Variability')
                ax2.set_ylabel('Standard Deviation')
                ax2.set_xticks(x_pos)
                ax2.set_xticklabels([label.split('\\n')[0] for label in group_labels])
                ax2.legend()
                ax2.grid(True, alpha=0.3)
        
        plt.suptitle('Stride Change Analysis: Success vs Failure by Age Group', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'stride_change_age_comparison.png', 'population')
    
    def plot_retention_analysis(self) -> List[Path]:
        """Plot retention analysis for invisible trial type."""
        plot_paths = []
        
        # Check if we have retention data
        retention_cols = [col for col in self.filtered_df.columns if 'retention' in col]
        if not retention_cols:
            print("No retention data found")
            return plot_paths
        
        # Plot 1: Retention by clamp type
        try:
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            
            # Box plot comparing clamp1 vs clamp2
            ax = axes[0]
            retention_data = []
            labels = []
            
            for clamp in ['clamp1', 'clamp2']:
                col = f'invis_retention_{clamp}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        retention_data.append(data)
                        labels.append(clamp.capitalize())
            
            if retention_data:
                bp = ax.boxplot(retention_data, labels=labels, patch_artist=True)
                colors = ['lightblue', 'lightcoral']
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Retention')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Retention')
                ax.set_title('Retention by Clamp Period')
                ax.set_ylabel('Retention')
                ax.legend()
                ax.grid(True, alpha=0.3)
            
            # Retention vs age
            ax = axes[1]
            for clamp, color in [('clamp1', 'blue'), ('clamp2', 'red')]:
                col = f'invis_retention_{clamp}_const'
                if col in self.filtered_df.columns:
                    valid_data = self.filtered_df[['age', col]].dropna()
                    if not valid_data.empty:
                        ax.scatter(valid_data['age'], valid_data[col], 
                                 alpha=0.6, s=60, color=color, label=clamp.capitalize())
            
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            ax.axhline(y=1, color='green', linestyle='--', alpha=0.5)
            ax.set_xlabel('Age (years)')
            ax.set_ylabel('Retention')
            ax.set_title('Retention vs Age')
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            # Overall retention histogram
            ax = axes[2]
            if 'invis_retention_overall_const' in self.filtered_df.columns:
                data = self.filtered_df['invis_retention_overall_const'].dropna()
                if not data.empty:
                    ax.hist(data, bins=20, alpha=0.7, edgecolor='black')
                    ax.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='No Retention')
                    ax.axvline(x=1, color='green', linestyle='--', alpha=0.7, label='Perfect Retention')
                    ax.axvline(x=data.mean(), color='blue', linestyle='-', alpha=0.7, 
                              label=f'Mean = {data.mean():.3f}')
            
            ax.set_xlabel('Overall Retention')
            ax.set_ylabel('Frequency')
            ax.set_title('Overall Retention Distribution')
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            plt.suptitle('Retention Analysis: Success Clamp Performance\n(Retention = 0: No retention, Retention = 1: Perfect retention)', 
                         fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            plot_path = self.config.population_plots_dir / 'retention_analysis.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close()
            plot_paths.append(plot_path)
            
        except Exception as e:
            print(f"Failed to create retention analysis plot: {e}")
        
        return plot_paths
    
    # Add this method to plot learning distributions
    def plot_learning_distributions(self) -> Path:
        """Plot learning metric distributions by trial and condition."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Plot by trial type (combining conditions)
        for i, trial in enumerate(trials):
            ax = axes[0, i]
            learning_data = []
            labels = []
            colors = []
            
            for condition in conditions:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        learning_data.append(data)
                        labels.append(f'{condition.capitalize()}')
                        colors.append('lightblue' if condition == 'max' else 'lightcoral')
            
            if learning_data:
                bp = ax.boxplot(learning_data, labels=labels, patch_artist=True)
                for patch, color in zip(bp['boxes'], colors):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                # Add reference lines
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Learning')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Learning')
            
            ax.set_title(f'{trial.upper()} Trial')
            ax.set_ylabel('Learning')
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.legend()
        
        # Plot by condition (combining trials)  
        for j, condition in enumerate(conditions):
            ax = axes[1, j]
            learning_data = []
            labels = []
            colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
            
            for trial in trials:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    if not data.empty:
                        learning_data.append(data)
                        labels.append(trial.upper())
            
            if learning_data:
                bp = ax.boxplot(learning_data, labels=labels, patch_artist=True)
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                
                # Add reference lines
                ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Learning')
                ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Learning')
            
            ax.set_title(f'{condition.capitalize()} Target Condition')
            ax.set_ylabel('Learning')
            ax.grid(True, alpha=0.3)
            if j == 0:
                ax.legend()
        
        # Overall distribution in bottom right
        ax = axes[1, 2]
        all_learning_data = []
        for trial in trials:
            for condition in conditions:
                col = f'{trial}_learning_{condition}_const'
                if col in self.filtered_df.columns:
                    data = self.filtered_df[col].dropna()
                    all_learning_data.extend(data.values)
        
        if all_learning_data:
            ax.hist(all_learning_data, bins=30, alpha=0.7, edgecolor='black')
            ax.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='No Learning')
            ax.axvline(x=1, color='green', linestyle='--', alpha=0.7, label='Perfect Learning')
            ax.axvline(x=np.mean(all_learning_data), color='blue', linestyle='-', alpha=0.7, 
                      label=f'Mean = {np.mean(all_learning_data):.3f}')
        
        ax.set_title('Overall Learning Distribution')
        ax.set_xlabel('Learning')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.suptitle('Learning Metric Distributions\nLearning = 0: No adaptation, Learning = 1: Perfect adaptation to target', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'learning_distributions.png', 'population')
    
    def plot_age_vs_learning(self) -> Path:
        """Plot age vs learning by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate shared axis limits
        learning_cols = [f'{trial}_learning_{condition}_const' for trial in trials for condition in conditions]
        available_cols = [col for col in learning_cols if col in self.filtered_df.columns]
        
        if available_cols:
            all_learning_data = []
            for col in available_cols:
                data = self.filtered_df[col].dropna()
                if not data.empty:
                    all_learning_data.extend(data.values)
            
            if all_learning_data:
                y_min = min(all_learning_data)
                y_max = max(all_learning_data)
                y_range = y_max - y_min
                y_padding = y_range * 0.05
                y_limits = [y_min - y_padding, y_max + y_padding]
            else:
                y_limits = None
        else:
            y_limits = None
        
        # Calculate age limits
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min = age_data.min()
            age_max = age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_learning_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Learning')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.grid(True, alpha=0.3)
                        
                        # Add reference line at learning = 0 (no learning)
                        ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='No Learning')
                        # Add reference line at learning = 1 (perfect learning)
                        ax.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='Perfect Learning')
                        ax.legend()
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                
                # Set consistent axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
                if y_limits:
                    ax.set_ylim(y_limits)
        
        plt.suptitle('Age vs Learning by Trial/Condition\nLearning = (avg_stride - preferred_stride) / (target - preferred_stride)', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_learning.png', 'population')

    def plot_trial_patterns_by_age(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """
        Plot trial type patterns (vis1, invis, vis2) for each age group with shared y-axis.
        """
        
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'mean_success_rates' in results]
        
        if not valid_groups:
            return None
        
        fig, axes = plt.subplots(1, len(valid_groups), figsize=(5*len(valid_groups), 6))
        if len(valid_groups) == 1:
            axes = [axes]
        
        trial_order = ['vis1', 'invis', 'vis2']
        
        # Calculate shared y-axis limits
        all_means = []
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            for trial in trial_order:
                if trial in group_data['mean_success_rates']:
                    mean_val = group_data['mean_success_rates'][trial]
                    if not np.isnan(mean_val):
                        all_means.append(mean_val)
        
        if all_means:
            y_min = min(all_means)
            y_max = max(all_means)
            y_range = y_max - y_min
            y_padding = y_range * 0.1  # 10% padding
            shared_ylim = [max(0, y_min - y_padding), min(1, y_max + y_padding)]
        else:
            shared_ylim = [0, 1]
        
        for i, group in enumerate(valid_groups):
            ax = axes[i]
            group_data = age_results['group_analyses'][group]
            
            # Get mean success rates in order
            means = []
            for trial in trial_order:
                if trial in group_data['mean_success_rates']:
                    means.append(group_data['mean_success_rates'][trial])
                else:
                    means.append(np.nan)
            
            # Plot line with markers
            valid_indices = [j for j, val in enumerate(means) if not np.isnan(val)]
            valid_trials = [trial_order[j] for j in valid_indices]
            valid_means = [means[j] for j in valid_indices]
            
            if valid_means:
                ax.plot(valid_trials, valid_means, 'o-', linewidth=3, markersize=10, alpha=0.8,
                       color=self.colors['primary'])
                
                # Add value labels
                for j, (trial, mean) in enumerate(zip(valid_trials, valid_means)):
                    ax.text(j, mean + (shared_ylim[1] - shared_ylim[0]) * 0.02, 
                           f'{mean:.3f}', ha='center', va='bottom', 
                           fontweight='bold', fontsize=11)
            
            ax.set_ylim(shared_ylim)  # Apply shared y-axis limits
            ax.set_ylabel('Mean Success Rate')
            ax.set_xlabel('Trial Type')
            ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})')
            ax.grid(True, alpha=0.3)
            
            # Add significance indicator
            if 'trial_type_effect' in group_data:
                sig_text = "Significant trial effect" if group_data['trial_type_effect']['significant'] else "No significant effect"
                p_val = group_data['trial_type_effect']['p_value']
                ax.text(0.5, 0.95, f'{sig_text}\np = {p_val:.3f}', 
                       transform=ax.transAxes, ha='center', va='top',
                       bbox=dict(boxstyle='round,pad=0.3', 
                                facecolor='lightgreen' if group_data['trial_type_effect']['significant'] else 'lightcoral', 
                                alpha=0.7))
        
        plt.suptitle('Trial Type Patterns by Age Group', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to: {save_path}")
        
        return fig
    
    def plot_trial_patterns_msl_by_age(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """
        Plot mean stride length trial type patterns (vis1, invis, vis2) for each age group 
        showing both max and min conditions with different colors on the same plots.
        Now includes shaded error regions showing ±1 standard deviation.
        """
        
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results]
        
        if not valid_groups:
            return None
        
        # Calculate mean MSL AND standard deviations by trial type and condition for each age group
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            
            # Get age range for this group
            age_groups = age_results.get('age_group_definitions', {})
            if group in age_groups:
                min_age, max_age = age_groups[group]
                
                # Filter data for this age group
                age_mask = (self.filtered_df['age'] >= min_age) & (self.filtered_df['age'] < max_age)
                group_df = self.filtered_df[age_mask]
                
                # Calculate mean AND std MSL for each trial type and condition separately
                mean_msl_max = {}
                std_msl_max = {}
                mean_msl_min = {}
                std_msl_min = {}
                
                for trial in ['vis1', 'invis', 'vis2']:
                    # Max condition
                    max_col = f'{trial}_msl_max_const'
                    if max_col in group_df.columns:
                        values = group_df[max_col].dropna()
                        if len(values) > 0:
                            mean_msl_max[trial] = values.mean()
                            std_msl_max[trial] = values.std()
                        else:
                            mean_msl_max[trial] = np.nan
                            std_msl_max[trial] = np.nan
                    else:
                        mean_msl_max[trial] = np.nan
                        std_msl_max[trial] = np.nan
                    
                    # Min condition
                    min_col = f'{trial}_msl_min_const'
                    if min_col in group_df.columns:
                        values = group_df[min_col].dropna()
                        if len(values) > 0:
                            mean_msl_min[trial] = values.mean()
                            std_msl_min[trial] = values.std()
                        else:
                            mean_msl_min[trial] = np.nan
                            std_msl_min[trial] = np.nan
                    else:
                        mean_msl_min[trial] = np.nan
                        std_msl_min[trial] = np.nan
                
                group_data['mean_msl_max_by_trial'] = mean_msl_max
                group_data['std_msl_max_by_trial'] = std_msl_max
                group_data['mean_msl_min_by_trial'] = mean_msl_min
                group_data['std_msl_min_by_trial'] = std_msl_min
        
        fig, axes = plt.subplots(1, len(valid_groups), figsize=(5*len(valid_groups), 6))
        if len(valid_groups) == 1:
            axes = [axes]
        
        trial_order = ['vis1', 'invis', 'vis2']
        
        # Calculate shared y-axis limits across both conditions (including error ranges)
        all_values = []
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            for trial in trial_order:
                # Check both max and min values including error ranges
                if trial in group_data.get('mean_msl_max_by_trial', {}):
                    mean_val = group_data['mean_msl_max_by_trial'][trial]
                    std_val = group_data.get('std_msl_max_by_trial', {}).get(trial, 0)
                    if not np.isnan(mean_val) and not np.isnan(std_val):
                        all_values.extend([mean_val - std_val, mean_val + std_val])
                
                if trial in group_data.get('mean_msl_min_by_trial', {}):
                    mean_val = group_data['mean_msl_min_by_trial'][trial]
                    std_val = group_data.get('std_msl_min_by_trial', {}).get(trial, 0)
                    if not np.isnan(mean_val) and not np.isnan(std_val):
                        all_values.extend([mean_val - std_val, mean_val + std_val])
        
        if all_values:
            y_min = min(all_values)
            y_max = max(all_values)
            y_range = y_max - y_min
            y_padding = y_range * 0.1  # 10% padding
            shared_ylim = [y_min - y_padding, y_max + y_padding]
        else:
            shared_ylim = [0, 1]  # Default range
        
        for i, group in enumerate(valid_groups):
            ax = axes[i]
            group_data = age_results['group_analyses'][group]
            
            # Get mean stride lengths and standard deviations for both conditions
            max_means = []
            max_stds = []
            min_means = []
            min_stds = []
            
            for trial in trial_order:
                # Max condition
                if trial in group_data.get('mean_msl_max_by_trial', {}):
                    max_means.append(group_data['mean_msl_max_by_trial'][trial])
                    max_stds.append(group_data.get('std_msl_max_by_trial', {}).get(trial, 0))
                else:
                    max_means.append(np.nan)
                    max_stds.append(np.nan)
                
                # Min condition
                if trial in group_data.get('mean_msl_min_by_trial', {}):
                    min_means.append(group_data['mean_msl_min_by_trial'][trial])
                    min_stds.append(group_data.get('std_msl_min_by_trial', {}).get(trial, 0))
                else:
                    min_means.append(np.nan)
                    min_stds.append(np.nan)
            
            # Convert to numpy arrays for easier manipulation
            max_means = np.array(max_means)
            max_stds = np.array(max_stds)
            min_means = np.array(min_means)
            min_stds = np.array(min_stds)
            
            # Create x-axis positions
            x_positions = np.arange(len(trial_order))
            
            # Plot lines with shaded error regions for both conditions
            # Max condition (larger target)
            valid_max_mask = ~(np.isnan(max_means) | np.isnan(max_stds))
            if valid_max_mask.any():
                valid_x = x_positions[valid_max_mask]
                valid_max_means = max_means[valid_max_mask]
                valid_max_stds = max_stds[valid_max_mask]
                
                # Plot line
                ax.plot(valid_x, valid_max_means, 'o-', linewidth=3, markersize=10, 
                       alpha=0.8, color='#2E8B57', label='Max Target')  # Sea green
                
                # Add shaded error region (±1 SD)
                ax.fill_between(valid_x, 
                               valid_max_means - valid_max_stds,
                               valid_max_means + valid_max_stds,
                               color='#2E8B57', alpha=0.3)
                
                # Add value labels for max (mean ± std)
                for j, (x, mean, std) in enumerate(zip(valid_x, valid_max_means, valid_max_stds)):
                    ax.text(x, mean + (shared_ylim[1] - shared_ylim[0]) * 0.04, 
                           f'{mean:.3f}±{std:.3f}', ha='center', va='bottom', 
                           fontweight='bold', fontsize=9, color='#2E8B57')
            
            # Min condition (smaller target)
            valid_min_mask = ~(np.isnan(min_means) | np.isnan(min_stds))
            if valid_min_mask.any():
                valid_x = x_positions[valid_min_mask]
                valid_min_means = min_means[valid_min_mask]
                valid_min_stds = min_stds[valid_min_mask]
                
                # Plot line
                ax.plot(valid_x, valid_min_means, 's-', linewidth=3, markersize=10, 
                       alpha=0.8, color='#B22222', label='Min Target')  # Fire brick red
                
                # Add shaded error region (±1 SD)
                ax.fill_between(valid_x, 
                               valid_min_means - valid_min_stds,
                               valid_min_means + valid_min_stds,
                               color='#B22222', alpha=0.3)
                
                # Add value labels for min (mean ± std, offset slightly to avoid overlap)
                for j, (x, mean, std) in enumerate(zip(valid_x, valid_min_means, valid_min_stds)):
                    ax.text(x, mean - (shared_ylim[1] - shared_ylim[0]) * 0.04, 
                           f'{mean:.3f}±{std:.3f}', ha='center', va='top', 
                           fontweight='bold', fontsize=9, color='#B22222')
            
            ax.set_ylim(shared_ylim)  # Apply shared y-axis limits
            ax.set_ylabel('Mean Stride Length')
            ax.set_xlabel('Trial Type')
            ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")}, '
                        f'age={group_data.get("descriptive_stats", {}).get("age_info", {}).get("mean_age", "?"):.1f}y)')
            ax.grid(True, alpha=0.3)
            ax.legend(loc='upper right')
            
            # Set x-axis labels
            ax.set_xticks(x_positions)
            ax.set_xticklabels([trial.upper() for trial in trial_order])
            
            # Add significance indicator if available
            if 'trial_type_effect_msl' in group_data:
                sig_text = "Significant trial effect" if group_data['trial_type_effect_msl']['significant'] else "No significant effect"
                p_val = group_data['trial_type_effect_msl']['p_value']
                ax.text(0.5, 0.05, f'{sig_text}\np = {p_val:.3f}', 
                       transform=ax.transAxes, ha='center', va='bottom',
                       bbox=dict(boxstyle='round,pad=0.3', 
                                facecolor='lightgreen' if group_data['trial_type_effect_msl']['significant'] else 'lightcoral', 
                                alpha=0.7))
        
        plt.suptitle('Mean Stride Length Patterns by Age Group\n(Max vs Min Target Conditions with ±1 SD)', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"MSL patterns figure with error shading saved to: {save_path}")
        
        return fig
    
    def plot_trial_patterns_sd_by_age(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """
        Plot stride length variability (SD) trial type patterns (vis1, invis, vis2) for each age group with shared y-axis.
        """
        
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'mean_sd_by_trial' in results]
        
        if not valid_groups:
            # If no mean_sd_by_trial data, calculate it from the filtered dataframe
            valid_groups = [name for name, results in age_results['group_analyses'].items() 
                           if 'error' not in results]
            
            # Calculate mean SD by trial type for each age group
            for group in valid_groups:
                group_data = age_results['group_analyses'][group]
                
                # Get age range for this group
                age_groups = age_results.get('age_group_definitions', {})
                if group in age_groups:
                    min_age, max_age = age_groups[group]
                    
                    # Filter data for this age group
                    age_mask = (self.filtered_df['age'] >= min_age) & (self.filtered_df['age'] < max_age)
                    group_df = self.filtered_df[age_mask]
                    
                    # Calculate mean SD for each trial type
                    mean_sd_values = {}
                    for trial in ['vis1', 'invis', 'vis2']:
                        for condition in ['max', 'min']:
                            col = f'{trial}_sd_{condition}_const'
                            if col in group_df.columns:
                                values = group_df[col].dropna()
                                if len(values) > 0:
                                    if trial not in mean_sd_values:
                                        mean_sd_values[trial] = []
                                    mean_sd_values[trial].append(values.mean())
                    
                    # Average across conditions for each trial type
                    for trial in mean_sd_values:
                        if mean_sd_values[trial]:
                            mean_sd_values[trial] = np.mean(mean_sd_values[trial])
                        else:
                            mean_sd_values[trial] = np.nan
                    
                    group_data['mean_sd_by_trial'] = mean_sd_values
        
        if not valid_groups:
            return None
        
        fig, axes = plt.subplots(1, len(valid_groups), figsize=(5*len(valid_groups), 6))
        if len(valid_groups) == 1:
            axes = [axes]
        
        trial_order = ['vis1', 'invis', 'vis2']
        
        # Calculate shared y-axis limits
        all_means = []
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            for trial in trial_order:
                if trial in group_data.get('mean_sd_by_trial', {}):
                    mean_val = group_data['mean_sd_by_trial'][trial]
                    if not np.isnan(mean_val):
                        all_means.append(mean_val)
        
        if all_means:
            y_min = min(all_means)
            y_max = max(all_means)
            y_range = y_max - y_min
            y_padding = y_range * 0.1  # 10% padding
            shared_ylim = [y_min - y_padding, y_max + y_padding]
        else:
            shared_ylim = [0, 1]  # Default range
        
        for i, group in enumerate(valid_groups):
            ax = axes[i]
            group_data = age_results['group_analyses'][group]
            
            # Get mean stride variabilities in order
            means = []
            for trial in trial_order:
                if trial in group_data.get('mean_sd_by_trial', {}):
                    means.append(group_data['mean_sd_by_trial'][trial])
                else:
                    means.append(np.nan)
            
            # Plot line with markers
            valid_indices = [j for j, val in enumerate(means) if not np.isnan(val)]
            valid_trials = [trial_order[j] for j in valid_indices]
            valid_means = [means[j] for j in valid_indices]
            
            if valid_means:
                ax.plot(valid_trials, valid_means, 'o-', linewidth=3, markersize=10, alpha=0.8,
                       color=self.colors['warning'])  # Different color for SD
                
                # Add value labels
                for j, (trial, mean) in enumerate(zip(valid_trials, valid_means)):
                    ax.text(j, mean + (shared_ylim[1] - shared_ylim[0]) * 0.02, 
                           f'{mean:.3f}', ha='center', va='bottom', 
                           fontweight='bold', fontsize=11)
            
            ax.set_ylim(shared_ylim)  # Apply shared y-axis limits
            ax.set_ylabel('Stride Length Variability (SD)')
            ax.set_xlabel('Trial Type')
            ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})')
            ax.grid(True, alpha=0.3)
            
            # Add significance indicator if available
            if 'trial_type_effect_sd' in group_data:
                sig_text = "Significant trial effect" if group_data['trial_type_effect_sd']['significant'] else "No significant effect"
                p_val = group_data['trial_type_effect_sd']['p_value']
                ax.text(0.5, 0.95, f'{sig_text}\np = {p_val:.3f}', 
                       transform=ax.transAxes, ha='center', va='top',
                       bbox=dict(boxstyle='round,pad=0.3', 
                                facecolor='lightgreen' if group_data['trial_type_effect_sd']['significant'] else 'lightcoral', 
                                alpha=0.7))
        
        plt.suptitle('Stride Length Variability Patterns by Age Group', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"SD patterns figure saved to: {save_path}")
        
        return fig
    
    def plot_age_stratified_results(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """
        Create comprehensive visualization of age-stratified ANOVA results.
        
        Parameters:
        -----------
        age_results : Dict
            Results from run_age_stratified_anova()
        save_path : str, optional
            Path to save the figure
        
        Returns:
        --------
        matplotlib Figure object
        """
        
        # Extract valid groups
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'trial_type_effect' in results]
        
        if not valid_groups:
            print("No valid age groups found for visualization")
            return None
        
        # Set up the figure
        n_groups = len(valid_groups)
        fig = plt.figure(figsize=(20, 16))  # Increased height from 12 to 16
        
        # Create subplot grid: 3 rows x n_groups columns, plus summary row
        # Set up the figure with 5 rows instead of 4
        gs = fig.add_gridspec(5, max(n_groups, 3), hspace=0.5, wspace=0.3, 
                             height_ratios=[1, 1, 1.3, 1.2, 1.5])  # Increased last ratio from 1 to 1.5
        
        # Color scheme for trial types
        trial_colors = {'vis1': '#1f77b4', 'invis': '#ff7f0e', 'vis2': '#2ca02c'}
        
        # Row 1: Mean success rates by age group and trial type
        for i, group in enumerate(valid_groups):
            ax = fig.add_subplot(gs[0, i])
            group_data = age_results['group_analyses'][group]
            
            if 'mean_success_rates' in group_data:
                trials = list(group_data['mean_success_rates'].keys())
                means = list(group_data['mean_success_rates'].values())
                colors = [trial_colors.get(trial, 'gray') for trial in trials]
                
                bars = ax.bar(trials, means, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
                
                # Add value labels on bars
                for bar, mean in zip(bars, means):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{mean:.2f}', ha='center', va='bottom', fontweight='bold')
                
                ax.set_ylim(0, 1)
                ax.set_ylabel('Mean Success Rate')
                ax.set_title(f'{group.title()}\n(n={group_data.get("descriptive_stats", {}).get("n_subjects", "?")})')
                ax.grid(True, alpha=0.3)
                
                # Add age info
                age_info = group_data.get('descriptive_stats', {}).get('age_info', {})
                if 'mean_age' in age_info:
                    ax.text(0.5, 0.95, f'Mean age: {age_info["mean_age"]:.1f}y', 
                           transform=ax.transAxes, ha='center', va='top', 
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.7))
        
        # Row 2: Effect sizes (eta-squared) comparison
        ax_effect = fig.add_subplot(gs[1, :n_groups])
        
        effect_sizes = []
        group_names = []
        significance = []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                group_names.append(group.title())
                significance.append(group_data['trial_type_effect']['significant'])
        
        if effect_sizes:
            colors = ['green' if sig else 'red' for sig in significance]
            bars = ax_effect.bar(group_names, effect_sizes, color=colors, alpha=0.7, edgecolor='black')
            
            # Add significance indicators
            for i, (bar, sig, eta2) in enumerate(zip(bars, significance, effect_sizes)):
                height = bar.get_height()
                sig_text = '***' if sig else 'ns'
                ax_effect.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                              f'{eta2:.3f}\n{sig_text}', ha='center', va='bottom', fontweight='bold')
            
            ax_effect.set_ylabel('Effect Size (η²)')
            ax_effect.set_title('Trial Type Effect Sizes by Age Group')
            ax_effect.grid(True, alpha=0.3)
            
            # Add effect size interpretation lines
            ax_effect.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='Small (0.01)')
            ax_effect.axhline(y=0.06, color='orange', linestyle='--', alpha=0.5, label='Medium (0.06)')
            ax_effect.axhline(y=0.14, color='red', linestyle='--', alpha=0.5, label='Large (0.14)')
            ax_effect.legend(loc='upper right')
        
        # Row 3: F-statistics comparison
        ax_f = fig.add_subplot(gs[2, :n_groups])
        
        f_stats = []
        p_values = []
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                f_stats.append(group_data['trial_type_effect']['F'])
                p_values.append(group_data['trial_type_effect']['p_value'])
        
        if f_stats:
            colors = ['green' if p < 0.05 else 'red' for p in p_values]
            bars = ax_f.bar(group_names, f_stats, color=colors, alpha=0.7, edgecolor='black')
            
            # Add p-value labels
            for i, (bar, f_val, p_val) in enumerate(zip(bars, f_stats, p_values)):
                height = bar.get_height()
                p_text = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'
                # Reduce the vertical offset if bars are tall
                text_offset = min(0.2, max(f_stats) * 0.05) if f_stats else 0.2
                ax_f.text(bar.get_x() + bar.get_width()/2., height + text_offset,
                         f'F={f_val:.1f}\n{p_text}', ha='center', va='bottom', fontweight='bold')
            
            ax_f.set_ylabel('F-statistic')
            ax_f.set_title('Trial Type F-statistics by Age Group')
            ax_f.grid(True, alpha=0.3)
            
            # Adjust y-axis limits to accommodate text labels
            max_f = max(f_stats) if f_stats else 1
            ax_f.set_ylim(0, max_f * 1.3)  # Add 30% extra space for labels
        
        # Row 4: Dedicated Post-hoc Comparisons Visualization
        ax_posthoc = fig.add_subplot(gs[3, :])
        
        # Get actual comparison names from the data
        all_comparison_names = set()
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                all_comparison_names.update(group_data['posthoc_comparisons'].keys())
        
        comparison_names = sorted(list(all_comparison_names))
        comparison_labels = [name.replace('_vs_', ' vs ').upper() for name in comparison_names]
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c'][:len(comparison_names)]
        
        if comparison_names and valid_groups:
            # Create grouped bar plot
            x_positions = np.arange(len(valid_groups))
            bar_width = 0.25
            
            for i, (comp_name, comp_label, color) in enumerate(zip(comparison_names, comparison_labels, colors)):
                effect_sizes = []
                significances = []
                p_values = []
                
                for group in valid_groups:
                    group_data = age_results['group_analyses'][group]
                    if ('posthoc_comparisons' in group_data and 
                        comp_name in group_data['posthoc_comparisons']):
                        comp_data = group_data['posthoc_comparisons'][comp_name]
                        effect_sizes.append(comp_data['cohens_d'])
                        significances.append(comp_data['significant'])
                        p_values.append(comp_data['p_corrected'])
                    else:
                        effect_sizes.append(0.0)
                        significances.append(False)
                        p_values.append(1.0)
                
                # Create bars for ALL groups
                bars = ax_posthoc.bar(x_positions + i * bar_width, 
                                     [abs(es) for es in effect_sizes], 
                                     bar_width, label=comp_label, color=color, alpha=0.7)
                
                # Add labels for ALL bars
                for j, (bar, es, sig, p_val) in enumerate(zip(bars, effect_sizes, significances, p_values)):
                    height = bar.get_height()
                    
                    if sig:
                        if p_val < 0.001:
                            sig_symbol = "***"
                        elif p_val < 0.01:
                            sig_symbol = "**"
                        else:
                            sig_symbol = "*"
                        text_color = 'black'
                        text_weight = 'bold'
                    else:
                        sig_symbol = "ns"
                        text_color = 'gray'
                        text_weight = 'normal'
                    
                    if height > 0.01 or es != 0:
                        ax_posthoc.text(bar.get_x() + bar.get_width()/2., max(height + 0.02, 0.05),
                                       f'd={es:.2f}\n{sig_symbol}', 
                                       ha='center', va='bottom', fontsize=9, 
                                       color=text_color, fontweight=text_weight)
                    else:
                        ax_posthoc.text(bar.get_x() + bar.get_width()/2., 0.05,
                                       f'No data', 
                                       ha='center', va='bottom', fontsize=8, 
                                       color='red', style='italic')
            
            ax_posthoc.set_xlabel('Age Groups')
            ax_posthoc.set_ylabel('Effect Size (|Cohen\'s d|)')
            ax_posthoc.set_title('Post-Hoc Pairwise Comparisons: Effect Sizes by Age Group')
            ax_posthoc.set_xticks(x_positions + bar_width)
            ax_posthoc.set_xticklabels([g.title() for g in valid_groups])
            ax_posthoc.legend(loc='upper right')
            ax_posthoc.grid(True, alpha=0.3)
            
            # Add effect size interpretation lines
            ax_posthoc.axhline(y=0.2, color='gray', linestyle='--', alpha=0.5, label='Small (0.2)')
            ax_posthoc.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Medium (0.5)')
            ax_posthoc.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='Large (0.8)')
            
            # Set reasonable y-axis limits
            all_effects = []
            for group in valid_groups:
                group_data = age_results['group_analyses'][group]
                if 'posthoc_comparisons' in group_data:
                    for comp_data in group_data['posthoc_comparisons'].values():
                        all_effects.append(abs(comp_data['cohens_d']))
            
            max_effect = max(all_effects) if all_effects else 0.5
            ax_posthoc.set_ylim(0, max(max_effect * 1.4, 0.3))
        
        # Row 5: Enhanced Summary table with proper column widths
        ax_table = fig.add_subplot(gs[4, :])
        ax_table.axis('off')
        
        # Create enhanced summary table with post-hoc information
        table_data = []
        headers = ['Age Group', 'N', 'Mean Age', 'F-stat', 'p-value', 'η²', 'Significant', 'Key Comparisons']
        
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'trial_type_effect' in group_data and 'descriptive_stats' in group_data:
                trial_effect = group_data['trial_type_effect']
                desc_stats = group_data['descriptive_stats']
                
                eta2 = trial_effect['effect_size_eta2']
                if eta2 < 0.01:
                    interpretation = 'Negligible'
                elif eta2 < 0.06:
                    interpretation = 'Small'
                elif eta2 < 0.14:
                    interpretation = 'Medium'
                else:
                    interpretation = 'Large'
                
                # Extract key post-hoc comparisons in single line format
                posthoc_summary = "No comparisons"
                if 'posthoc_comparisons' in group_data:
                    posthoc = group_data['posthoc_comparisons']
                    comparison_summaries = []
                    
                    # Show ALL comparisons in compact single-line format
                    for comp_name, comp_data in posthoc.items():
                        clean_name = comp_name.replace('_vs_', '>')
                        d_val = comp_data['cohens_d']
                        
                        if comp_data['significant']:
                            sig_marker = "*"
                        else:
                            sig_marker = "ns"
                        
                        # Compact single-line format
                        comparison_summaries.append(f"{clean_name}(d={d_val:.2f},{sig_marker})")
                    
                    if comparison_summaries:
                        # Join with semicolons for single line
                        posthoc_summary = '; '.join(comparison_summaries)
                    else:
                        posthoc_summary = "No comparisons calculated"
                
                row = [
                    group.title(),
                    str(desc_stats['n_subjects']),
                    f"{desc_stats['age_info']['mean_age']:.1f}y",
                    f"{trial_effect['F']:.2f}",
                    f"{trial_effect['p_value']:.3f}" if trial_effect['p_value'] >= 0.001 else "<0.001",
                    f"{eta2:.3f}",
                    "Yes" if trial_effect['significant'] else "No",
                    posthoc_summary
                ]
                table_data.append(row)
        
        if table_data:
            # Create table with custom column widths
            table = ax_table.table(cellText=table_data,
                                  colLabels=headers,
                                  cellLoc='center',
                                  loc='center',
                                  bbox=[0, 0, 1, 1])
            
            # Set font size and scaling first
            table.auto_set_font_size(False)
            table.set_fontsize(8)  # Smaller font to fit more text
            table.scale(1, 2)  # Reduced row height since we're using single lines now
            
            # Set custom column widths - make Key Comparisons column much wider
            col_widths = [0.12, 0.08, 0.10, 0.10, 0.10, 0.10, 0.12, 0.28]  # Key Comparisons gets 28% of width
            
            # Apply column widths using the correct matplotlib method
            cellDict = table.get_celld()
            for i, width in enumerate(col_widths):
                for j in range(len(table_data) + 1):  # +1 for header row
                    if (j, i) in cellDict:
                        cellDict[(j, i)].set_width(width)
            
            # Set alignment for all cells
            for i in range(len(table_data)):
                for j in range(len(headers)):
                    if (i+1, j) in cellDict:  # +1 for header row
                        cell = cellDict[(i+1, j)]
                        cell.get_text().set_verticalalignment('center')
                        cell.get_text().set_horizontalalignment('center')
                        # Special handling for Key Comparisons column - smaller font
                        if j == 7:  # Key Comparisons column
                            cell.get_text().set_fontsize(7)  # Smaller font for this column
            
            # Color code significance
            for i in range(len(table_data)):
                if table_data[i][6] == "Yes":  # Significant
                    table[(i+1, 6)].set_facecolor('#90EE90')  # Light green
                else:
                    table[(i+1, 6)].set_facecolor('#FFB6C1')  # Light red
                
                # Color code post-hoc results
                posthoc_text = table_data[i][7]
                if "No comparisons" in posthoc_text:
                    table[(i+1, 7)].set_facecolor('#FFE4B5')  # Light orange - no data
                elif "*" in posthoc_text:
                    table[(i+1, 7)].set_facecolor('#E0FFE0')  # Very light green - has significant
                elif "ns" in posthoc_text:
                    table[(i+1, 7)].set_facecolor('#F0F8FF')  # Very light blue - has non-significant
                else:
                    table[(i+1, 7)].set_facecolor('#FFFFFF')  # White - default
            
            # Style header
            for j in range(len(headers)):
                header_cell = table[(0, j)]
                header_cell.set_facecolor('#4CAF50')
                header_cell.set_text_props(weight='bold', color='white')
                # Make header text smaller to fit
                header_cell.get_text().set_fontsize(9)
        
        plt.suptitle('Age-Stratified ANOVA Results: Trial Type Effects', fontsize=16, fontweight='bold')
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Figure saved to: {save_path}")
        
        return fig

    
    def plot_posthoc_heatmap(self, age_results: Dict, save_path: str = None) -> plt.Figure:
        """
        Create a dedicated heatmap visualization of post-hoc comparisons across age groups.
        """
        
        valid_groups = [name for name, results in age_results['group_analyses'].items() 
                       if 'error' not in results and 'posthoc_comparisons' in results]
        
        if not valid_groups:
            print("No valid age groups with post-hoc data found")
            return None
        
        all_comparison_names = set()
        for group in valid_groups:
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                all_comparison_names.update(group_data['posthoc_comparisons'].keys())
        
        comparison_names = sorted(list(all_comparison_names))  # Use actual names from data
        comparison_labels = [name.replace('_vs_', ' vs ').upper() for name in comparison_names]
        
        print(f"Heatmap using comparison names: {comparison_names}")  # Debug output
        
        if not comparison_names:
            print("No comparison names found in data")
            return None
        
        # Create figure with 3 subplots: effect sizes, p-values, and summary
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Post-Hoc Comparison Analysis Across Age Groups', fontsize=16, fontweight='bold')
        
        # Prepare data matrices
        effect_matrix = np.zeros((len(comparison_names), len(valid_groups)))
        p_value_matrix = np.ones((len(comparison_names), len(valid_groups)))
        sig_matrix = np.zeros((len(comparison_names), len(valid_groups)), dtype=bool)
        
        for i, group in enumerate(valid_groups):
            group_data = age_results['group_analyses'][group]
            if 'posthoc_comparisons' in group_data:
                posthoc = group_data['posthoc_comparisons']
                
                for j, comp_name in enumerate(comparison_names):
                    if comp_name in posthoc:
                        comp_data = posthoc[comp_name]
                        effect_matrix[j, i] = comp_data['cohens_d']
                        p_value_matrix[j, i] = comp_data['p_corrected']
                        sig_matrix[j, i] = comp_data['significant']
        
        # Plot 1: Effect sizes heatmap
        ax1 = axes[0, 0]
        im1 = ax1.imshow(effect_matrix, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax1.set_xticks(range(len(valid_groups)))
        ax1.set_xticklabels([g.title() for g in valid_groups])
        ax1.set_yticks(range(len(comparison_labels)))
        ax1.set_yticklabels(comparison_labels)
        ax1.set_title('Effect Sizes (Cohen\'s d)')
        
        # Add text annotations for ALL effect sizes (not just significant ones)
        for i in range(len(comparison_names)):
            for j in range(len(valid_groups)):
                text = f'{effect_matrix[i, j]:.2f}'
                color = 'white' if abs(effect_matrix[i, j]) > 0.5 else 'black'
                weight = 'bold' if abs(effect_matrix[i, j]) > 0.3 else 'normal'  # Bold for medium+ effects
                ax1.text(j, i, text, ha='center', va='center', color=color, fontweight=weight)
        
        plt.colorbar(im1, ax=ax1, label='Cohen\'s d')
        
        # Plot 2: P-values heatmap
        ax2 = axes[0, 1]
        # Use log scale for p-values for better visualization
        log_p_matrix = -np.log10(np.maximum(p_value_matrix, 1e-10))  # Avoid log(0)
        im2 = ax2.imshow(log_p_matrix, cmap='Reds', aspect='auto')
        ax2.set_xticks(range(len(valid_groups)))
        ax2.set_xticklabels([g.title() for g in valid_groups])
        ax2.set_yticks(range(len(comparison_labels)))
        ax2.set_yticklabels(comparison_labels)
        ax2.set_title('Statistical Significance (-log10(p))')
        
        # Add text annotations for ALL p-values 
        for i in range(len(comparison_names)):
            for j in range(len(valid_groups)):
                p_val = p_value_matrix[i, j]
                if p_val < 0.001:
                    text = '***'
                elif p_val < 0.01:
                    text = '**'
                elif p_val < 0.05:
                    text = '*'
                else:
                    text = f'{p_val:.2f}'  # Show actual p-value for non-significant
                
                color = 'white' if log_p_matrix[i, j] > 1 else 'black'
                weight = 'bold' if p_val < 0.05 else 'normal'
                ax2.text(j, i, text, ha='center', va='center', color=color, fontweight=weight)
        
        plt.colorbar(im2, ax=ax2, label='-log10(p-value)')
        
        # Plot 3: Combined effect size and significance scatter plot - SHOW ALL POINTS
        ax3 = axes[1, 0]
        
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Match the bar chart colors
        
        for i, (comp_name, comp_label, color) in enumerate(zip(comparison_names, comparison_labels, colors)):
            group_effects = effect_matrix[i, :]
            group_p_values = p_value_matrix[i, :]
            group_sigs = sig_matrix[i, :]
            
            # Plot ALL points regardless of significance
            for j, (effect, p_val, sig) in enumerate(zip(group_effects, group_p_values, group_sigs)):
                if sig:
                    marker = 'o'  # Circle for significant
                    alpha = 0.8
                    size = 100
                    edge_color = 'black'
                    edge_width = 2
                else:
                    marker = '^'  # Triangle for non-significant  
                    alpha = 0.6
                    size = 80
                    edge_color = 'gray'
                    edge_width = 1
                
                ax3.scatter(effect, -np.log10(max(p_val, 1e-10)), 
                           color=color, marker=marker, s=size, alpha=alpha,
                           edgecolors=edge_color, linewidths=edge_width,
                           label=f'{comp_label}' if j == 0 else "")  # Only label once per comparison
        
        # Always add reference lines
        ax3.axhline(y=-np.log10(0.05), color='red', linestyle='--', alpha=0.7, label='p=0.05')
        ax3.axhline(y=-np.log10(0.01), color='orange', linestyle='--', alpha=0.5, label='p=0.01')
        ax3.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        ax3.axvline(x=0.2, color='gray', linestyle=':', alpha=0.5, label='Small effect')
        ax3.axvline(x=-0.2, color='gray', linestyle=':', alpha=0.5)
        ax3.axvline(x=0.5, color='orange', linestyle=':', alpha=0.5, label='Medium effect')
        ax3.axvline(x=-0.5, color='orange', linestyle=':', alpha=0.5)
        
        ax3.set_xlabel('Effect Size (Cohen\'s d)')
        ax3.set_ylabel('Statistical Significance (-log10(p))')
        ax3.set_title('Effect Size vs Statistical Significance (All Comparisons)')
        ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax3.grid(True, alpha=0.3)
        
        # Set reasonable axis limits to show all data
        all_effects = effect_matrix.flatten()
        all_log_p = -np.log10(np.maximum(p_value_matrix.flatten(), 1e-10))
        
        x_margin = (np.max(all_effects) - np.min(all_effects)) * 0.1
        y_margin = (np.max(all_log_p) - np.min(all_log_p)) * 0.1
        
        ax3.set_xlim(np.min(all_effects) - x_margin, np.max(all_effects) + x_margin)
        ax3.set_ylim(-0.1, np.max(all_log_p) + y_margin)
        
        # Plot 4: Summary statistics table
        ax4 = axes[1, 1]
        ax4.axis('off')
        
        # Create summary data
        summary_data = []
        headers = ['Comparison', 'Mean |d|', 'Max |d|', 'N Significant', 'N Groups']
        
        for i, comp_label in enumerate(comparison_labels):
            effects = np.abs(effect_matrix[i, :])
            sigs = sig_matrix[i, :]
            
            summary_data.append([
                comp_label,
                f'{np.mean(effects):.2f}',
                f'{np.max(effects):.2f}',
                f'{np.sum(sigs)}/{len(valid_groups)}',
                str(len(valid_groups))
            ])
        
        table = ax4.table(cellText=summary_data,
                         colLabels=headers,
                         cellLoc='center',
                         loc='center',
                         bbox=[0, 0.3, 1, 0.6])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        
        # Style the table
        for j in range(len(headers)):
            table[(0, j)].set_facecolor('#4CAF50')
            table[(0, j)].set_text_props(weight='bold', color='white')
        
        ax4.set_title('Summary Statistics', pad=20)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Post-hoc heatmap saved to: {save_path}")
        
        return fig

    
    
    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate age limits (consistent across all plots)
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min = age_data.min()
            age_max = age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_sr_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Success Rate')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                
                # Set consistent age axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
        
        plt.suptitle('Age vs Success Rates by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig,'age_vs_success_rates.png', 'population')
    
    def plot_age_vs_mean_stride_length(self) -> Path:
        """Plot age vs mean stride length by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate shared axis limits
        msl_cols = [f'{trial}_msl_{condition}_const' for trial in trials for condition in conditions]
        available_cols = [col for col in msl_cols if col in self.filtered_df.columns]
        
        if available_cols:
            all_msl_data = []
            for col in available_cols:
                data = self.filtered_df[col].dropna()
                if not data.empty:
                    all_msl_data.extend(data.values)
            
            if all_msl_data:
                y_min = min(all_msl_data)
                y_max = max(all_msl_data)
                y_range = y_max - y_min
                y_padding = y_range * 0.05
                y_limits = [y_min - y_padding, y_max + y_padding]
            else:
                y_limits = None
        else:
            y_limits = None
        
        # Calculate age limits
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min = age_data.min()
            age_max = age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_msl_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Mean Stride Length')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                
                # Set consistent axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
                if y_limits:
                    ax.set_ylim(y_limits)
        
        plt.suptitle('Age vs Mean Stride Length by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig,'age_vs_mean_stride_length.png', 'population')
    
    def plot_age_vs_stride_variability(self) -> Path:
        """Plot age vs stride length variability by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        # Calculate shared axis limits
        sd_cols = [f'{trial}_sd_{condition}_const' for trial in trials for condition in conditions]
        available_cols = [col for col in sd_cols if col in self.filtered_df.columns]
        
        if available_cols:
            all_sd_data = []
            for col in available_cols:
                data = self.filtered_df[col].dropna()
                if not data.empty:
                    all_sd_data.extend(data.values)
            
            if all_sd_data:
                y_min = min(all_sd_data)
                y_max = max(all_sd_data)
                y_range = y_max - y_min
                y_padding = y_range * 0.05
                y_limits = [y_min - y_padding, y_max + y_padding]
            else:
                y_limits = None
        else:
            y_limits = None
        
        # Calculate age limits
        age_data = self.filtered_df['age'].dropna()
        if not age_data.empty:
            age_min = age_data.min()
            age_max = age_data.max()
            age_range = age_max - age_min
            age_padding = age_range * 0.05
            x_limits = [age_min - age_padding, age_max + age_padding]
        else:
            x_limits = None
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_sd_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Stride Length Variability (SD)')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                
                # Set consistent axis limits
                if x_limits:
                    ax.set_xlim(x_limits)
                if y_limits:
                    ax.set_ylim(y_limits)
        
        plt.suptitle('Age vs Stride Length Variability by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig,'age_vs_stride_variability.png', 'population')
    
    def plot_age_vs_success_rates(self) -> Path:
        """Plot age vs success rates by trial/condition with motor noise coloring."""
        trials = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        has_motor_noise = 'mot_noise' in self.filtered_df.columns
        
        fig, axes = plt.subplots(len(trials), len(conditions), figsize=(12, 4*len(trials)))
        if len(trials) == 1:
            axes = axes.reshape(1, -1)
        
        for i, trial in enumerate(trials):
            for j, condition in enumerate(conditions):
                ax = axes[i, j]
                col = f'{trial}_sr_{condition}_const'
                
                if col in self.filtered_df.columns:
                    cols_to_use = ['age', col] + (['mot_noise'] if has_motor_noise else [])
                    valid_data = self.filtered_df[cols_to_use].dropna()
                    
                    if not valid_data.empty:
                        if has_motor_noise:
                            scatter = ax.scatter(valid_data['age'], valid_data[col], 
                                               c=valid_data['mot_noise'], cmap='plasma', 
                                               alpha=0.7, s=60, edgecolors='white', linewidths=0.5)
                            if i == 0 and j == len(conditions) - 1:
                                cbar = plt.colorbar(scatter, ax=ax)
                                cbar.set_label('Motor Noise', rotation=270, labelpad=15)
                        else:
                            ax.scatter(valid_data['age'], valid_data[col], alpha=0.7, s=60)
                        
                        self.add_trendline(ax, valid_data['age'], valid_data[col])
                        
                        ax.set_xlabel('Age (years)')
                        ax.set_ylabel('Success Rate')
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
                        ax.set_ylim(-0.05, 1.05)
                        ax.grid(True, alpha=0.3)
                    else:
                        ax.text(0.5, 0.5, 'No Data', ha='center', va='center',
                               transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{trial.upper()}: {condition.capitalize()} Target')
        
        plt.suptitle('Age vs Success Rates by Trial/Condition', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_success_rates.png', 'population')
    
    def plot_correlation_matrix(self) -> Path:
        """Plot correlation matrix of key variables."""
        key_cols = ['age']
        if 'mot_noise' in self.filtered_df.columns:
            key_cols.append('mot_noise')
        if 'pref_asymmetry' in self.filtered_df.columns:
            key_cols.append('pref_asymmetry')
        
        # Add success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        key_cols.extend(sr_cols[:6])  # Limit to first 6 to avoid clutter
        
        if len(key_cols) > 1:
            fig = plt.figure(figsize=(12, 10))
            
            corr_matrix = self.filtered_df[key_cols].corr()
            mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
            
            sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                       square=True, linewidths=0.5, fmt='.2f')
            
            plt.title('Correlation Matrix of Key Variables', fontsize=16, fontweight='bold')
            plt.tight_layout()
            
            return self.save_figure(fig, 'correlation_matrix.png', 'population')
    
    def plot_trial_comparison(self) -> Path:
        """Plot comparison across trial types."""
        trial_cols = []
        for trial in ['vis1', 'invis', 'vis2']:
            for condition in ['max', 'min']:
                col = f'{trial}_sr_{condition}_const'
                if col in self.filtered_df.columns:
                    trial_cols.append(col)
        
        if len(trial_cols) < 2:
            return None
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 6))
        
        # Create boxplot
        plot_data = []
        labels = []
        for col in trial_cols:
            data = self.filtered_df[col].dropna()
            if len(data) > 0:
                plot_data.append(data)
                # Clean up labels
                parts = col.split('_')
                trial_name = parts[0].upper()
                condition_name = parts[2].capitalize()
                labels.append(f'{trial_name}\n{condition_name}')
        
        if plot_data:
            ax.boxplot(plot_data, labels=labels)
            ax.set_ylabel('Success Rate')
            ax.set_title('Success Rate Distribution by Trial Type and Condition')
            ax.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
        
        plt.tight_layout()
        return self.save_figure(fig, 'trial_comparison.png', 'population')
    
    def plot_anova_results(self) -> List[Path]:
        """Create comprehensive plots for ANOVA results."""
        plot_paths = []
        
        # 1. Success Rate ANOVA visualization
        success_plot = self.plot_success_rate_anova()
        if success_plot:
            plot_paths.append(success_plot)
        
        # 2. Mean Stride Length ANOVA visualization
        msl_plot = self.plot_mean_stride_length_anova()
        if msl_plot:
            plot_paths.append(msl_plot)
        
        # 3. Stride Variability ANOVA visualization
        variability_plot = self.plot_stride_variability_anova()
        if variability_plot:
            plot_paths.append(variability_plot)
        
        # 4. Combined comparison plot
        combined_plot = self.plot_combined_anova_comparison()
        if combined_plot:
            plot_paths.append(combined_plot)
        
        # 5. Age vs ANOVA covariates with connected lines
        age_covariate_plots = self.plot_age_vs_anova_covariates()
        plot_paths.extend(age_covariate_plots)
        
        return plot_paths
    
    def plot_success_rate_anova(self) -> Optional[Path]:
        """Plot success rates for ANOVA visualization."""
        # Get success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        if not sr_cols:
            return None
        
        # Prepare data for plotting
        plot_data = []
        trial_types = []
        conditions = []
        
        for col in sr_cols:
            parts = col.split('_')
            if len(parts) >= 3:
                trial_type = parts[0]
                condition = parts[2]
                data = self.filtered_df[col].dropna()
                
                for value in data:
                    plot_data.append(value)
                    trial_types.append(trial_type.upper())
                    conditions.append(condition.capitalize())
        
        if not plot_data:
            return None
        
        # Create DataFrame for plotting
        df_plot = pd.DataFrame({
            'success_rate': plot_data,
            'trial_type': trial_types,
            'condition': conditions
        })
        
        # Create the plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Box plot by trial type
        trial_order = ['VIS1', 'INVIS', 'VIS2']
        trial_data = [df_plot[df_plot['trial_type'] == t]['success_rate'].values 
                     for t in trial_order if t in df_plot['trial_type'].values]
        trial_labels = [t for t in trial_order if t in df_plot['trial_type'].values]
        
        if trial_data:
            bp1 = ax1.boxplot(trial_data, labels=trial_labels, patch_artist=True)
            colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
            for patch, color in zip(bp1['boxes'], colors[:len(bp1['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax1.set_ylabel('Success Rate')
        ax1.set_title('Success Rate by Trial Type')
        ax1.grid(True, alpha=0.3)
        
        # Box plot by condition
        condition_order = ['Max', 'Min']
        condition_data = [df_plot[df_plot['condition'] == c]['success_rate'].values 
                         for c in condition_order if c in df_plot['condition'].values]
        condition_labels = [c for c in condition_order if c in df_plot['condition'].values]
        
        if condition_data:
            bp2 = ax2.boxplot(condition_data, labels=condition_labels, patch_artist=True)
            colors = [self.colors['success'], self.colors['warning']]
            for patch, color in zip(bp2['boxes'], colors[:len(bp2['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax2.set_ylabel('Success Rate')
        ax2.set_title('Success Rate by Target Condition')
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle('Success Rate ANOVA Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'success_rate_anova.png', 'population')
    
    def plot_mean_stride_length_anova(self) -> Optional[Path]:
        """Plot mean stride length for ANOVA visualization."""
        # Get mean stride length columns
        msl_cols = [col for col in self.filtered_df.columns if '_msl_' in col and '_const' in col]
        if not msl_cols:
            return None
        
        # Prepare data for plotting
        plot_data = []
        trial_types = []
        conditions = []
        
        for col in msl_cols:
            parts = col.split('_')
            if len(parts) >= 3:
                trial_type = parts[0]
                condition = parts[2]
                data = self.filtered_df[col].dropna()
                
                for value in data:
                    plot_data.append(value)
                    trial_types.append(trial_type.upper())
                    conditions.append(condition.capitalize())
        
        if not plot_data:
            return None
        
        # Create DataFrame for plotting
        df_plot = pd.DataFrame({
            'mean_stride_length': plot_data,
            'trial_type': trial_types,
            'condition': conditions
        })
        
        # Create the plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Box plot by trial type
        trial_order = ['VIS1', 'INVIS', 'VIS2']
        trial_data = [df_plot[df_plot['trial_type'] == t]['mean_stride_length'].values 
                     for t in trial_order if t in df_plot['trial_type'].values]
        trial_labels = [t for t in trial_order if t in df_plot['trial_type'].values]
        
        if trial_data:
            bp1 = ax1.boxplot(trial_data, labels=trial_labels, patch_artist=True)
            colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
            for patch, color in zip(bp1['boxes'], colors[:len(bp1['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax1.set_ylabel('Mean Stride Length')
        ax1.set_title('Mean Stride Length by Trial Type')
        ax1.grid(True, alpha=0.3)
        
        # Box plot by condition
        condition_order = ['Max', 'Min']
        condition_data = [df_plot[df_plot['condition'] == c]['mean_stride_length'].values 
                         for c in condition_order if c in df_plot['condition'].values]
        condition_labels = [c for c in condition_order if c in df_plot['condition'].values]
        
        if condition_data:
            bp2 = ax2.boxplot(condition_data, labels=condition_labels, patch_artist=True)
            colors = [self.colors['success'], self.colors['warning']]
            for patch, color in zip(bp2['boxes'], colors[:len(bp2['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax2.set_ylabel('Mean Stride Length')
        ax2.set_title('Mean Stride Length by Target Condition')
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle('Mean Stride Length ANOVA Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'mean_stride_length_anova.png', 'population')
    
    def plot_stride_variability_anova(self) -> Optional[Path]:
        """Plot stride length variability for ANOVA visualization."""
        # Get stride variability columns
        sd_cols = [col for col in self.filtered_df.columns if '_sd_' in col and '_const' in col]
        if not sd_cols:
            return None
        
        # Prepare data for plotting
        plot_data = []
        trial_types = []
        conditions = []
        
        for col in sd_cols:
            parts = col.split('_')
            if len(parts) >= 3:
                trial_type = parts[0]
                condition = parts[2]
                data = self.filtered_df[col].dropna()
                
                for value in data:
                    plot_data.append(value)
                    trial_types.append(trial_type.upper())
                    conditions.append(condition.capitalize())
        
        if not plot_data:
            return None
        
        # Create DataFrame for plotting
        df_plot = pd.DataFrame({
            'stride_variability': plot_data,
            'trial_type': trial_types,
            'condition': conditions
        })
        
        # Create the plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Box plot by trial type
        trial_order = ['VIS1', 'INVIS', 'VIS2']
        trial_data = [df_plot[df_plot['trial_type'] == t]['stride_variability'].values 
                     for t in trial_order if t in df_plot['trial_type'].values]
        trial_labels = [t for t in trial_order if t in df_plot['trial_type'].values]
        
        if trial_data:
            bp1 = ax1.boxplot(trial_data, labels=trial_labels, patch_artist=True)
            colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
            for patch, color in zip(bp1['boxes'], colors[:len(bp1['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax1.set_ylabel('Stride Length Variability (SD)')
        ax1.set_title('Stride Variability by Trial Type')
        ax1.grid(True, alpha=0.3)
        
        # Box plot by condition
        condition_order = ['Max', 'Min']
        condition_data = [df_plot[df_plot['condition'] == c]['stride_variability'].values 
                         for c in condition_order if c in df_plot['condition'].values]
        condition_labels = [c for c in condition_order if c in df_plot['condition'].values]
        
        if condition_data:
            bp2 = ax2.boxplot(condition_data, labels=condition_labels, patch_artist=True)
            colors = [self.colors['success'], self.colors['warning']]
            for patch, color in zip(bp2['boxes'], colors[:len(bp2['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax2.set_ylabel('Stride Length Variability (SD)')
        ax2.set_title('Stride Variability by Target Condition')
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle('Stride Length Variability ANOVA Results', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'stride_variability_anova.png', 'population')
    
    def plot_age_vs_anova_covariates(self) -> List[Path]:
        """Create plots showing age vs each ANOVA dependent variable with connected lines."""
        plot_paths = []
        
        # 1. Age vs Success Rates
        success_plot = self.plot_age_vs_success_rates_connected()
        if success_plot:
            plot_paths.append(success_plot)
        
        # 2. Age vs Stride Variability
        variability_plot = self.plot_age_vs_stride_variability_connected()
        if variability_plot:
            plot_paths.append(variability_plot)
        
        return plot_paths
    
    def plot_age_vs_success_rates_connected(self) -> Optional[Path]:
        """Plot age vs success rates with connected lines for max/min conditions."""
        # Get success rate columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        if not sr_cols:
            return None
        
        # Organize data by trial type
        trial_types = ['vis1', 'invis', 'vis2']
        trial_colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for i, (trial_type, color) in enumerate(zip(trial_types, trial_colors)):
            ax = axes[i]
            
            # Get max and min columns for this trial type
            max_col = f'{trial_type}_sr_max_const'
            min_col = f'{trial_type}_sr_min_const'
            
            if max_col in self.filtered_df.columns and min_col in self.filtered_df.columns:
                # Get data for subjects with both max and min values
                plot_data = self.filtered_df[['age', max_col, min_col]].dropna()
                
                if not plot_data.empty:
                    # Plot connected lines for each subject
                    for _, row in plot_data.iterrows():
                        age = row['age']
                        max_sr = row[max_col]
                        min_sr = row[min_col]
                        
                        # Plot line connecting max and min for this subject
                        ax.plot([age, age], [max_sr, min_sr], 
                               color=color, alpha=0.3, linewidth=1)
                        
                        # Plot points
                        ax.scatter(age, max_sr, color=color, s=60, alpha=0.8, 
                                 marker='o', label='Max Target' if row.name == plot_data.index[0] else "")
                        ax.scatter(age, min_sr, color=color, s=60, alpha=0.8, 
                                 marker='s', label='Min Target' if row.name == plot_data.index[0] else "")
                    
                    # Add trendlines
                    self.add_trendline(ax, plot_data['age'], plot_data[max_col])
                    
                    # Customize plot
                    ax.set_xlabel('Age (years)')
                    ax.set_ylabel('Success Rate')
                    ax.set_title(f'{trial_type.upper()} Trial')
                    ax.set_ylim(-0.05, 1.05)
                    ax.grid(True, alpha=0.3)
                    ax.legend(loc='upper right')
        
        plt.suptitle('Age vs Success Rates: Max and Min Conditions Connected', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_success_rates_connected.png', 'population')
    
    def plot_age_vs_mean_stride_length_connected(self) -> Optional[Path]:
        """Plot age vs mean stride length with connected lines for max/min conditions."""
        # Get mean stride length columns
        msl_cols = [col for col in self.filtered_df.columns if '_msl_' in col and '_const' in col]
        if not msl_cols:
            return None
        
        # Organize data by trial type
        trial_types = ['vis1', 'invis', 'vis2']
        trial_colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for i, (trial_type, color) in enumerate(zip(trial_types, trial_colors)):
            ax = axes[i]
            
            # Get max and min columns for this trial type
            max_col = f'{trial_type}_msl_max_const'
            min_col = f'{trial_type}_msl_min_const'
            
            if max_col in self.filtered_df.columns and min_col in self.filtered_df.columns:
                # Get data for subjects with both max and min values
                plot_data = self.filtered_df[['age', max_col, min_col]].dropna()
                
                if not plot_data.empty:
                    # Plot connected lines for each subject
                    for _, row in plot_data.iterrows():
                        age = row['age']
                        max_msl = row[max_col]
                        min_msl = row[min_col]
                        
                        # Plot line connecting max and min for this subject
                        ax.plot([age, age], [max_msl, min_msl], 
                               color=color, alpha=0.3, linewidth=1)
                        
                        # Plot points
                        ax.scatter(age, max_msl, color=color, s=60, alpha=0.8, 
                                 marker='o', label='Max Target' if row.name == plot_data.index[0] else "")
                        ax.scatter(age, min_msl, color=color, s=60, alpha=0.8, 
                                 marker='s', label='Min Target' if row.name == plot_data.index[0] else "")
                    
                    # Add trendlines for both conditions
                    self.add_trendline(ax, plot_data['age'], plot_data[max_col])
                    
                    # Customize plot
                    ax.set_xlabel('Age (years)')
                    ax.set_ylabel('Mean Stride Length')
                    ax.set_title(f'{trial_type.upper()} Trial')
                    ax.grid(True, alpha=0.3)
                    ax.legend()
        
        plt.suptitle('Age vs Mean Stride Length: Max and Min Conditions Connected', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_mean_stride_length_connected.png', 'population')
    
    def plot_age_vs_stride_variability_connected(self) -> Optional[Path]:
        """Plot age vs stride variability with connected lines for max/min conditions."""
        # Get stride variability columns
        sd_cols = [col for col in self.filtered_df.columns if '_sd_' in col and '_const' in col]
        if not sd_cols:
            return None
        
        # Organize data by trial type
        trial_types = ['vis1', 'invis', 'vis2']
        trial_colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for i, (trial_type, color) in enumerate(zip(trial_types, trial_colors)):
            ax = axes[i]
            
            # Get max and min columns for this trial type
            max_col = f'{trial_type}_sd_max_const'
            min_col = f'{trial_type}_sd_min_const'
            
            if max_col in self.filtered_df.columns and min_col in self.filtered_df.columns:
                # Get data for subjects with both max and min values
                plot_data = self.filtered_df[['age', max_col, min_col]].dropna()
                
                if not plot_data.empty:
                    # Plot connected lines for each subject
                    for _, row in plot_data.iterrows():
                        age = row['age']
                        max_sd = row[max_col]
                        min_sd = row[min_col]
                        
                        # Plot line connecting max and min for this subject
                        ax.plot([age, age], [max_sd, min_sd], 
                               color=color, alpha=0.3, linewidth=1)
                        
                        # Plot points
                        ax.scatter(age, max_sd, color=color, s=60, alpha=0.8, 
                                 marker='o', label='Max Target' if row.name == plot_data.index[0] else "")
                        ax.scatter(age, min_sd, color=color, s=60, alpha=0.8, 
                                 marker='s', label='Min Target' if row.name == plot_data.index[0] else "")
                    
                    # Add trendlines
                    self.add_trendline(ax, plot_data['age'], plot_data[max_col])
                    
                    # Customize plot
                    ax.set_xlabel('Age (years)')
                    ax.set_ylabel('Stride Length Variability (SD)')
                    ax.set_title(f'{trial_type.upper()} Trial')
                    ax.grid(True, alpha=0.3)
                    ax.legend(loc='upper right')
        
        plt.suptitle('Age vs Stride Length Variability: Max and Min Conditions Connected', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'age_vs_stride_variability_connected.png', 'population')
    
    def plot_combined_anova_comparison(self) -> Optional[Path]:
        """Create a combined plot showing all three ANOVA measures."""
        # Get all relevant columns
        sr_cols = [col for col in self.filtered_df.columns if '_sr_' in col and '_const' in col]
        msl_cols = [col for col in self.filtered_df.columns if '_msl_' in col and '_const' in col]
        sd_cols = [col for col in self.filtered_df.columns if '_sd_' in col and '_const' in col]
        
        if not (sr_cols and msl_cols and sd_cols):
            return None
        
        # Create figure with subplots
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        
        # Plot success rates
        self._plot_anova_subplot(axes[0, 0], axes[0, 1], sr_cols, 'Success Rate', 'success_rate')
        
        # Plot mean stride length
        self._plot_anova_subplot(axes[1, 0], axes[1, 1], msl_cols, 'Mean Stride Length', 'mean_stride_length')
        
        # Plot stride variability
        self._plot_anova_subplot(axes[2, 0], axes[2, 1], sd_cols, 'Stride Variability (SD)', 'stride_variability')
        
        plt.suptitle('Comprehensive ANOVA Results Comparison', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        return self.save_figure(fig, 'combined_anova_comparison.png', 'population')
    
    def _plot_anova_subplot(self, ax1, ax2, cols, ylabel, value_name):
        """Helper function to create ANOVA subplot."""
        # Prepare data
        plot_data = []
        trial_types = []
        conditions = []
        
        for col in cols:
            parts = col.split('_')
            if len(parts) >= 3:
                trial_type = parts[0]
                condition = parts[2]
                data = self.filtered_df[col].dropna()
                
                for value in data:
                    plot_data.append(value)
                    trial_types.append(trial_type.upper())
                    conditions.append(condition.capitalize())
        
        if not plot_data:
            return
        
        df_plot = pd.DataFrame({
            value_name: plot_data,
            'trial_type': trial_types,
            'condition': conditions
        })
        
        # Plot by trial type
        trial_order = ['VIS1', 'INVIS', 'VIS2']
        trial_data = [df_plot[df_plot['trial_type'] == t][value_name].values 
                     for t in trial_order if t in df_plot['trial_type'].values]
        trial_labels = [t for t in trial_order if t in df_plot['trial_type'].values]
        
        if trial_data:
            bp1 = ax1.boxplot(trial_data, labels=trial_labels, patch_artist=True)
            colors = [self.colors['vis1'], self.colors['invis'], self.colors['vis2']]
            for patch, color in zip(bp1['boxes'], colors[:len(bp1['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax1.set_ylabel(ylabel)
        ax1.set_title(f'{ylabel} by Trial Type')
        ax1.grid(True, alpha=0.3)
        
        # Plot by condition
        condition_order = ['Max', 'Min']
        condition_data = [df_plot[df_plot['condition'] == c][value_name].values 
                         for c in condition_order if c in df_plot['condition'].values]
        condition_labels = [c for c in condition_order if c in df_plot['condition'].values]
        
        if condition_data:
            bp2 = ax2.boxplot(condition_data, labels=condition_labels, patch_artist=True)
            colors = [self.colors['success'], self.colors['warning']]
            for patch, color in zip(bp2['boxes'], colors[:len(bp2['boxes'])]):
                patch.set_facecolor(color)
                patch.set_alpha(0.7)
        
        ax2.set_ylabel(ylabel)
        ax2.set_title(f'{ylabel} by Target Condition')
        ax2.grid(True, alpha=0.3)

class IndividualVisualizer(BaseVisualizer):
    """Handles individual participant visualizations."""
    
    def __init__(self, data_manager: 'DataManager', config: AnalysisConfig):
        super().__init__(config)
        self.data_manager = data_manager
    
    def plot_stride_changes(self, subject_id: str, trial_type: str) -> Optional[Path]:
        """Plot stride change distributions for a subject."""
        subject = self.data_manager.subjects.get(subject_id)
        if not subject:
            return None
        
        trial_dict = subject.trial_data.get(trial_type)
        if not trial_dict or trial_dict['data'] is None:
            return None
        
        df = trial_dict['data']
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Process for max and min conditions
        for ax, condition in zip([ax1, ax2], ['max', 'min']):
            period_data = self._get_period_data(df, condition)
            if period_data is not None:
                self._plot_stride_distribution(ax, period_data, condition)
            else:
                ax.text(0.5, 0.5, f'No {condition} data', ha='center', va='center',
                       transform=ax.transAxes)
            
            ax.set_title(f'{condition.capitalize()} Target')
            ax.set_xlabel('Change in Stride Length')
            ax.set_ylabel('Probability Density')
            ax.grid(True, alpha=0.3)
        
        plt.suptitle(f'Subject {subject_id} - {trial_type.upper()} Trial (Age: {subject.age:.1f})')
        plt.tight_layout()
        
        filename = f'stride_changes_{subject_id}_{trial_type}.png'
        return self.save_figure(fig, filename, 'individual')
    
    def _get_period_data(self, df: pd.DataFrame, condition: str) -> Optional[pd.DataFrame]:
        """Extract period data for plotting."""
        if 'Target size' not in df.columns or 'Constant' not in df.columns:
            return None
        
        min_target = df['Target size'].min()
        min_target_periods = df[df['Target size'] <= min_target + 0.001]
        
        if min_target_periods.empty:
            return None
        
        const_value = (min_target_periods['Constant'].max() if condition == 'max'
                      else min_target_periods['Constant'].min())
        
        period_data = min_target_periods[
            np.isclose(min_target_periods['Constant'], const_value, rtol=1e-5)
        ]
        
        return period_data.tail(20) if not period_data.empty else None
    
    def _plot_stride_distribution(self, ax, period_data: pd.DataFrame, condition: str):
        """Plot stride change distribution."""
        period_sorted = period_data.sort_values('Stride Number').copy()
        period_sorted['Delta'] = period_sorted['Sum of gains and steps'].diff().shift(-1)
        period_sorted = period_sorted[:-1]
        
        success_deltas = period_sorted[period_sorted['Success'] == 1]['Delta'].dropna()
        failure_deltas = period_sorted[period_sorted['Success'] == 0]['Delta'].dropna()
        
        # Plot distributions
        if len(success_deltas) > 0:
            self._plot_kde(ax, success_deltas, 'green', 'After Success')
        
        if len(failure_deltas) > 0:
            self._plot_kde(ax, failure_deltas, 'red', 'After Failure')
        
        # Add means
        if len(success_deltas) > 0:
            ax.axvline(success_deltas.mean(), color='darkgreen', linestyle='--', 
                      label=f'Success Mean: {success_deltas.mean():.3f}')
        
        if len(failure_deltas) > 0:
            ax.axvline(failure_deltas.mean(), color='darkred', linestyle='--',
                      label=f'Failure Mean: {failure_deltas.mean():.3f}')
        
        ax.axvline(0, color='black', linestyle='-', alpha=0.3)
        ax.legend()
    
    def _plot_kde(self, ax, data, color, label):
        """Plot KDE distribution."""
        try:
            if len(data) >= 2:
                kde = gaussian_kde(data)
                x_range = np.linspace(data.min() - 0.1, data.max() + 0.1, 200)
                y_vals = kde(x_range)
                ax.plot(x_range, y_vals, color=color, linewidth=3, alpha=0.8, label=label)
                ax.fill_between(x_range, y_vals, alpha=0.3, color=color)
            else:
                ax.axvline(data.iloc[0], color=color, alpha=0.7, linewidth=3, label=label)
        except Exception:
            # Fallback to histogram
            ax.hist(data, bins=10, alpha=0.7, color=color, label=label, density=True)
    
    def plot_subject_summary(self, subject_id: str) -> Optional[Path]:
        """Plot comprehensive summary for a subject."""
        subject = self.data_manager.subjects.get(subject_id)
        if not subject:
            return None
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Success rates across trials
        ax1 = axes[0, 0]
        trial_types = ['vis1', 'invis', 'vis2']
        conditions = ['max', 'min']
        
        x_pos = np.arange(len(trial_types))
        width = 0.35
        
        max_rates = []
        min_rates = []
        
        for trial in trial_types:
            trial_dict = subject.trial_data.get(trial)
            if trial_dict and trial_dict['data'] is not None:
                df = trial_dict['data']
                
                # Get success rates for both conditions
                max_data, _ = self.data_manager._get_period_data(df, 'max')
                min_data, _ = self.data_manager._get_period_data(df, 'min')
                
                max_rate = max_data['Success'].mean() if max_data is not None else 0
                min_rate = min_data['Success'].mean() if min_data is not None else 0
                
                max_rates.append(max_rate)
                min_rates.append(min_rate)
            else:
                max_rates.append(0)
                min_rates.append(0)
        
        ax1.bar(x_pos - width/2, max_rates, width, label='Max Target', alpha=0.8)
        ax1.bar(x_pos + width/2, min_rates, width, label='Min Target', alpha=0.8)
        ax1.set_xlabel('Trial Type')
        ax1.set_ylabel('Success Rate')
        ax1.set_title('Success Rates by Trial Type')
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels([t.upper() for t in trial_types])
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Motor noise visualization (if available)
        ax2 = axes[0, 1]
        pref_dict = subject.trial_data.get('pref')
        if pref_dict and pref_dict['data'] is not None:
            pref_df = pref_dict['data']
            if all(col in pref_df.columns for col in ['Right step length', 'Left step length']):
                right_steps = pref_df['Right step length']
                left_steps = pref_df['Left step length']
                
                ax2.plot(right_steps, label='Right Steps', alpha=0.7)
                ax2.plot(left_steps, label='Left Steps', alpha=0.7)
                ax2.set_xlabel('Stride Number')
                ax2.set_ylabel('Step Length')
                ax2.set_title('Preference Trial Step Lengths')
                ax2.legend()
                ax2.grid(True, alpha=0.3)
        else:
            ax2.text(0.5, 0.5, 'No Preference Data', ha='center', va='center',
                    transform=ax2.transAxes, fontsize=12)
        
        # Plot 3: Age comparison
        ax3 = axes[1, 0]
        all_ages = [s.age for s in self.data_manager.subjects.values()]
        ax3.hist(all_ages, bins=20, alpha=0.7, label='All Subjects')
        ax3.axvline(subject.age, color='red', linestyle='--', linewidth=2, 
                   label=f'This Subject (Age: {subject.age:.1f})')
        ax3.set_xlabel('Age (years)')
        ax3.set_ylabel('Frequency')
        ax3.set_title('Age Distribution')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Trial timeline
        ax4 = axes[1, 1]
        ax4.text(0.5, 0.5, f'Subject ID: {subject_id}\nAge: {subject.age:.1f} years\n'
                f'Session: {subject.metadata.get("Session Date", "Unknown")}', 
                ha='center', va='center', transform=ax4.transAxes, fontsize=12,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax4.set_title('Subject Information')
        ax4.axis('off')
        
        plt.suptitle(f'Subject {subject_id} Summary', fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        filename = f'subject_summary_{subject_id}.png'
        return self.save_figure(fig, filename, 'individual')

# ==============================================================================
# MAIN ANALYSIS INTERFACE
# ==============================================================================

class MotorLearningAnalysis:
    """Main analysis interface that coordinates all components."""
    
    def __init__(self, config: AnalysisConfig = None):
        self.config = config or AnalysisConfig()
        self.data_manager: Optional[DataManager] = None
        self.metrics_df: Optional[pd.DataFrame] = None
        self.statistical_analyzer: Optional[StatisticalAnalyzer] = None
        self.population_visualizer: Optional[PopulationVisualizer] = None
        self.individual_visualizer: Optional[IndividualVisualizer] = None
        self.results: Optional[Dict] = None
    
    def load_data(self, metadata_path: str, data_root_dir: str, 
                  force_reprocess: bool = False) -> 'MotorLearningAnalysis':
        """Load and process data."""
        self.data_manager = DataManager(
            metadata_path, data_root_dir, 
            self.config, force_reprocess
        )
        return self
    
    def filter_data(self, **kwargs) -> 'MotorLearningAnalysis':
        """Filter data based on criteria."""
        if self.data_manager:
            self.data_manager = self.data_manager.filter_subjects(**kwargs)
        return self
    
    def calculate_metrics(self) -> 'MotorLearningAnalysis':
        """Calculate metrics for all subjects."""
        if not self.data_manager:
            raise ValueError("Data not loaded. Call load_data() first.")
        
        self.metrics_df = self.data_manager.calculate_metrics()
        
        # Initialize analyzers with metrics
        self.statistical_analyzer = StatisticalAnalyzer(self.metrics_df, self.config)
        self.population_visualizer = PopulationVisualizer(self.metrics_df, self.config)
        self.individual_visualizer = IndividualVisualizer(self.data_manager, self.config)
        
        return self
    
    def run_analysis(self, include_visualizations: bool = True) -> 'MotorLearningAnalysis':
        """Run comprehensive analysis and store results."""
        if self.metrics_df is None:
            raise ValueError("Metrics not calculated. Call calculate_metrics() first.")
        
        self.results = {
            'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
            'n_subjects': len(self.metrics_df),
            'analyses': {}
        }
        
        # Statistical analyses
        if self.statistical_analyzer:
            try:
                self.results['analyses']['regression'] = self.statistical_analyzer.run_regression_analysis()
            except Exception as e:
                self.results['analyses']['regression'] = {'error': str(e)}


            
            try:
                self.results['analyses']['correlations'] = self.statistical_analyzer.run_correlation_analysis()
            except Exception as e:
                self.results['analyses']['correlations'] = {'error': str(e)}
        
        # Visualizations
        if include_visualizations and self.population_visualizer:
            self.results['visualizations'] = {
                'population': [],
                'individual': []
            }
            
            # Population plots
            try:
                plot_path = self.population_visualizer.plot_age_vs_success_rates()
                if plot_path:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create age vs success rates plot: {e}")
            
            try:
                plot_path = self.population_visualizer.plot_age_vs_mean_stride_length()
                if plot_path:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create age vs mean stride length plot: {e}")
            
            try:
                plot_path = self.population_visualizer.plot_age_vs_stride_variability()
                if plot_path:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create age vs stride variability plot: {e}")
            
            try:
                plot_path = self.population_visualizer.plot_correlation_matrix()
                if plot_path:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create correlation matrix: {e}")
            
            try:
                plot_path = self.population_visualizer.plot_trial_comparison()
                if plot_path:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create trial comparison plot: {e}")
            
            # ANOVA-specific plots
            try:
                anova_plots = self.population_visualizer.plot_anova_results()
                for plot_path in anova_plots:
                    self.results['visualizations']['population'].append(str(plot_path))
            except Exception as e:
                print(f"Failed to create ANOVA plots: {e}")
        
        # Generate report
        self._generate_report()
        
        return self  # Return self to maintain fluent interface
    
    def get_results(self) -> Dict:
        """Get analysis results."""
        return self.results
    
    def _generate_report(self):
        """Generate analysis report."""
        if not self.results:
            return
        
        report_path = self.config.reports_dir / f"analysis_report_{self.results['timestamp']}.json"
        
        # Convert to JSON-serializable format
        def make_serializable(obj):
            if isinstance(obj, (np.integer, np.floating)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, Path):
                return str(obj)
            elif pd.isna(obj):
                return None
            elif hasattr(obj, '__dict__') and not isinstance(obj, (dict, list, tuple)):
                return str(type(obj).__name__)
            return obj
        
        serializable_results = json.loads(
            json.dumps(self.results, default=make_serializable)
        )
        
        with open(report_path, 'w') as f:
            json.dump(serializable_results, f, indent=2)
        
        print(f"Report saved to: {report_path}")

# ==============================================================================
# CONVENIENCE FUNCTIONS
# ==============================================================================

def run_motor_learning_analysis(metadata_path: str, data_root_dir: str,
                               output_dir: str = 'motor_learning_output',
                               required_trials: List[str] = None) -> MotorLearningAnalysis:
    """Convenience function to run complete analysis."""
    
    # Create config
    config = AnalysisConfig(base_output_dir=Path(output_dir))
    
    # Initialize and run analysis
    analysis = (MotorLearningAnalysis(config)
                .load_data(metadata_path, data_root_dir)
                .filter_data(required_trial_types=required_trials or ['vis1', 'invis', 'vis2'])
                .calculate_metrics()
                .run_analysis())
    
    return analysis

# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

def main():
    """Example usage of the motor learning analysis pipeline."""
    
    # Configuration
    data_root_dir = 'muh_data/'
    metadata_path = 'muh_metadata.csv'
    
    print("Motor Learning Analysis Pipeline - Corrected Version")
    print("=" * 60)
    
    # Method 1: Quick analysis with convenience function
    print("\n1. Running quick analysis...")
    analysis = run_motor_learning_analysis(metadata_path, data_root_dir)
    
    # Method 2: Step-by-step with more control
    print("\n2. Running step-by-step analysis...")
    analysis = (MotorLearningAnalysis()
        .load_data(metadata_path, data_root_dir, force_reprocess=False)
        .filter_data(required_trial_types=['vis1', 'invis', 'vis2'])
        .calculate_metrics()
        .run_analysis())
    
    # Access components
    print("\n3. Accessing analysis components...")
    data_manager = analysis.data_manager
    metrics_df = analysis.metrics_df
    stats = analysis.statistical_analyzer
    pop_viz = analysis.population_visualizer
    ind_viz = analysis.individual_visualizer
    
    # Run specific analyses
    print("\n4. Running specific analyses...")
    regression_results = stats.run_regression_analysis()
    correlation_results = stats.run_correlation_analysis()
    
    # Print results
    print("\nRegression Results:")
    print(f"  R²: {regression_results['metrics']['r2']:.3f}")
    print(f"  RMSE: {regression_results['metrics']['rmse']:.3f}")
    print(f"  Sample size: {regression_results['metrics']['n_samples']}")
    print(f"  Feature importances: {regression_results['feature_importances']}")
    
    print("\nANOVA Results:")
    for effect, result in anova_results.items():
        if 'error' not in result:
            print(f"  {effect}: F={result['F']:.3f}, p={result['p_value']:.3f}")
        else:
            print(f"  {effect}: {result['error']}")
    
    print(f"\nDataset Summary:")
    print(f"  Total subjects: {len(metrics_df)}")
    print(f"  Subjects with motor noise data: {metrics_df['mot_noise'].notna().sum()}")
    print(f"  Age range: {metrics_df['age'].min():.1f} - {metrics_df['age'].max():.1f} years")
    
    # Generate individual plots for first few subjects
    print("\n5. Generating individual plots...")
    subject_ids = list(data_manager.subjects.keys())[:3]  # First 3 subjects
    for subject_id in subject_ids:
        try:
            plot_path = ind_viz.plot_subject_summary(subject_id)
            if plot_path:
                print(f"  Created summary plot for {subject_id}: {plot_path}")
        except Exception as e:
            print(f"  Failed to create plot for {subject_id}: {e}")
    
    print("\n6. Analysis complete!")
    print(f"  All outputs saved to: {analysis.config.base_output_dir}")
    print(f"  Results: {analysis.get_results()}")
    
    return analysis

if __name__ == "__main__":
    # Example usage
    print("Motor Learning Analysis Pipeline - Corrected Version")
    print("=" * 60)
    print("\nThis pipeline provides a comprehensive analysis of motor learning data.")
    print("\nKey features:")
    print("- Automated data loading and preprocessing")
    print("- Comprehensive statistical analysis (regression, ANOVA, correlations)")
    print("- Population and individual visualizations")
    print("- Robust error handling and data validation")
    print("- Fluent interface for easy chaining of operations")
    print("\nTo run analysis:")
    print("  analysis = main()")
    print("  # or")
    print("  analysis = run_motor_learning_analysis('metadata.csv', 'data_dir/')")
    print("\nTo access specific components:")
    print("  stats = analysis.statistical_analyzer")
    print("  data = analysis.data_manager")
    print("  metrics = analysis.metrics_df")
    print("  results = analysis.get_results()")
    
    # Uncomment to run the example
    # analysis = main()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

class AgeStratifiedANOVAVisualizer:
    """Enhanced visualizer for age-stratified ANOVA results - FIXED VERSION"""
    
    def __init__(self, config, colors=None):
        self.config = config
        # FIXED: Use existing PopulationVisualizer colors or provide defaults
        if colors is None:
            self.colors = {
                'primary': '#667eea',
                'secondary': '#764ba2',
                'success': '#28a745',
                'warning': '#ffc107',
                'danger': '#dc3545',
                'vis1': '#1f77b4',
                'invis': '#ff7f0e',
                'vis2': '#2ca02c',
                'younger': '#1f77b4',
                'middle': '#ff7f0e', 
                'older': '#2ca02c',
                'significant': '#28a745',
                'non_significant': '#dc3545'
            }
        else:
            # Use provided colors and add missing ones
            self.colors = colors.copy()
            # Add any missing color keys with defaults
            default_colors = {
                'significant': '#28a745',
                'non_significant': '#dc3545',
                'younger': '#1f77b4',
                'middle': '#ff7f0e',
                'older': '#2ca02c'
            }
            for key, color in default_colors.items():
                if key not in self.colors:
                    self.colors[key] = color
    
    def plot_unified_anova_summary(self, anova_results, save_path=None):
        """
        Create a comprehensive summary plot for unified age-stratified ANOVA results.
        
        Parameters:
        -----------
        anova_results : Dict
            Results from run_unified_age_stratified_anova()
        save_path : str, optional
            Path to save the figure
        
        Returns:
        --------
        matplotlib Figure object
        """
        
        # Extract valid groups
        valid_groups = [name for name, results in anova_results['group_analyses'].items() 
                       if 'error' not in results and 'trial_type_effect' in results]
        
        if not valid_groups:
            print("No valid age groups found for visualization")
            return None
        
        # Set up figure with 4 subplots in 2x2 grid
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'Age-Stratified ANOVA: {anova_results["dependent_variable"]}', 
                     fontsize=16, fontweight='bold')
        
        # 1. Effect sizes comparison (top-left)
        self._plot_effect_sizes(axes[0, 0], anova_results, valid_groups)
        
        # 2. F-statistics comparison (top-right)
        self._plot_f_statistics(axes[0, 1], anova_results, valid_groups)
        
        # 3. Sample sizes and age info (bottom-left)
        self._plot_sample_info(axes[1, 0], anova_results, valid_groups)
        
        # 4. Covariate effects heatmap (bottom-right)
        self._plot_covariate_heatmap(axes[1, 1], anova_results, valid_groups)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Unified ANOVA summary saved to: {save_path}")
        
        return fig
    
    def _plot_effect_sizes(self, ax, anova_results, valid_groups):
        """Plot effect sizes (eta-squared) with significance indicators"""
        
        effect_sizes = []
        p_values = []
        group_names = []
        
        for group in valid_groups:
            group_data = anova_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                p_values.append(group_data['trial_type_effect']['p_value'])
                group_names.append(group.title())
        
        # FIXED: Use colors that definitely exist
        colors = [self.colors['success'] if p < 0.05 else self.colors['danger'] 
                 for p in p_values]
        
        bars = ax.bar(group_names, effect_sizes, color=colors, alpha=0.7, edgecolor='black')
        
        # Add value labels and significance markers
        for i, (bar, eta2, p_val) in enumerate(zip(bars, effect_sizes, p_values)):
            height = bar.get_height()
            
            # Significance marker
            if p_val < 0.001:
                sig_text = "***"
            elif p_val < 0.01:
                sig_text = "**"
            elif p_val < 0.05:
                sig_text = "*"
            else:
                sig_text = "ns"
            
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                   f'{eta2:.3f}\n{sig_text}', ha='center', va='bottom', 
                   fontweight='bold', fontsize=10)
        
        # Add effect size interpretation lines
        ax.axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='Small (0.01)')
        ax.axhline(y=0.06, color='orange', linestyle='--', alpha=0.5, label='Medium (0.06)')
        ax.axhline(y=0.14, color='red', linestyle='--', alpha=0.5, label='Large (0.14)')
        
        ax.set_ylabel('Effect Size (η²)')
        ax.set_title('Trial Type Effect Sizes by Age Group')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    def _plot_f_statistics(self, ax, anova_results, valid_groups):
        """Plot F-statistics with p-value annotations"""
        
        f_stats = []
        p_values = []
        group_names = []
        
        for group in valid_groups:
            group_data = anova_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                f_stats.append(group_data['trial_type_effect']['F'])
                p_values.append(group_data['trial_type_effect']['p_value'])
                group_names.append(group.title())
        
        # FIXED: Use colors that definitely exist
        colors = [self.colors['success'] if p < 0.05 else self.colors['danger'] 
                 for p in p_values]
        
        bars = ax.bar(group_names, f_stats, color=colors, alpha=0.7, edgecolor='black')
        
        # Add p-value labels
        for i, (bar, f_val, p_val) in enumerate(zip(bars, f_stats, p_values)):
            height = bar.get_height()
            p_text = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'
            
            # FIXED: Better calculation of text offset
            max_height = max(f_stats) if f_stats else 1
            text_offset = max_height * 0.05
            
            ax.text(bar.get_x() + bar.get_width()/2., height + text_offset,
                   f'F={f_val:.1f}\n{p_text}', ha='center', va='bottom', 
                   fontweight='bold', fontsize=9)
        
        ax.set_ylabel('F-statistic')
        ax.set_title('Trial Type F-statistics by Age Group')
        ax.grid(True, alpha=0.3)
        
        # Adjust y-axis to accommodate labels
        max_f = max(f_stats) if f_stats else 1
        ax.set_ylim(0, max_f * 1.3)
    
    def _plot_sample_info(self, ax, anova_results, valid_groups):
        """Plot sample sizes and age information"""
        
        sample_sizes = []
        mean_ages = []
        group_names = []
        
        for group in valid_groups:
            group_data = anova_results['group_analyses'][group]
            sample_sizes.append(group_data['descriptive_stats']['n_subjects'])
            
            # Get mean age from group assignments
            if group in anova_results['group_assignments']:
                mean_ages.append(anova_results['group_assignments'][group]['mean_age'])
            else:
                mean_ages.append(np.nan)
            
            group_names.append(group.title())
        
        # Create twin axis for age
        ax2 = ax.twinx()
        
        # Plot sample sizes as bars
        bars = ax.bar(group_names, sample_sizes, alpha=0.7, 
                     color=self.colors.get('primary', '#667eea'), label='Sample Size')
        
        # Plot mean ages as line
        ax2.plot(group_names, mean_ages, 'ro-', linewidth=3, markersize=8, 
                color=self.colors.get('warning', '#ffc107'), label='Mean Age')
        
        # Add value labels
        for i, (bar, n, age) in enumerate(zip(bars, sample_sizes, mean_ages)):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                   f'n={n}', ha='center', va='bottom', fontweight='bold')
            
            if not np.isnan(age):
                ax2.text(i, age + 0.3, f'{age:.1f}y', ha='center', va='bottom', 
                        fontweight='bold', color=self.colors.get('warning', '#ffc107'))
        
        ax.set_ylabel('Sample Size', color=self.colors.get('primary', '#667eea'))
        ax2.set_ylabel('Mean Age (years)', color=self.colors.get('warning', '#ffc107'))
        ax.set_title('Sample Characteristics by Age Group')
        
        # Combine legends
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        
        ax.grid(True, alpha=0.3)
    
    def _plot_covariate_heatmap(self, ax, anova_results, valid_groups):
        """Plot heatmap of covariate effects"""
        
        # Extract covariate effects
        covariate_data = {}
        all_covariates = set()
        
        for group in valid_groups:
            group_data = anova_results['group_analyses'][group]
            covariate_data[group] = {}
            
            for key, value in group_data.items():
                if '_covariate_effect' in key and isinstance(value, dict):
                    covariate_name = key.replace('_covariate_effect', '')
                    all_covariates.add(covariate_name)
                    
                    # Use correlation value
                    covariate_data[group][covariate_name] = value.get('correlation', 0)
        
        if not all_covariates:
            ax.text(0.5, 0.5, 'No covariate effects\nto display', 
                   ha='center', va='center', transform=ax.transAxes, fontsize=12)
            ax.set_title('Covariate Effects')
            ax.axis('off')
            return
        
        # Create matrix for heatmap
        covariates_list = sorted(list(all_covariates))
        correlation_matrix = np.zeros((len(covariates_list), len(valid_groups)))
        
        for i, covariate in enumerate(covariates_list):
            for j, group in enumerate(valid_groups):
                correlation_matrix[i, j] = covariate_data[group].get(covariate, 0)
        
        # Create heatmap
        im = ax.imshow(correlation_matrix, cmap='RdBu_r', aspect='auto', 
                      vmin=-1, vmax=1)
        
        # Set ticks and labels
        ax.set_xticks(range(len(valid_groups)))
        ax.set_xticklabels([g.title() for g in valid_groups])
        ax.set_yticks(range(len(covariates_list)))
        ax.set_yticklabels(covariates_list)
        
        # Add correlation values as text
        for i in range(len(covariates_list)):
            for j in range(len(valid_groups)):
                correlation = correlation_matrix[i, j]
                color = 'white' if abs(correlation) > 0.5 else 'black'
                ax.text(j, i, f'{correlation:.2f}', ha='center', va='center', 
                       color=color, fontweight='bold')
        
        ax.set_title('Covariate Correlations')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax, shrink=0.8)
        cbar.set_label('Correlation with DV')
    
    def plot_dependent_variable_patterns(self, anova_results, filtered_df, save_path=None):
        """
        Plot the actual dependent variable patterns across age groups and trial types.
        FIXED: Better handling of different dependent variable types.
        """
        
        dependent_var = anova_results['dependent_variable']
        valid_groups = [name for name, results in anova_results['group_analyses'].items() 
                       if 'error' not in results]
        
        if not valid_groups:
            return None
        
        # Determine the pattern type based on dependent variable
        if 'success rate' in dependent_var.lower():
            return self._plot_success_rate_patterns(anova_results, filtered_df, valid_groups, save_path)
        elif 'learning' in dependent_var.lower():
            return self._plot_learning_patterns(anova_results, filtered_df, valid_groups, save_path)
        elif 'retention' in dependent_var.lower():
            return self._plot_retention_patterns(anova_results, filtered_df, valid_groups, save_path)
        elif 'stride' in dependent_var.lower():
            return self._plot_stride_patterns(anova_results, filtered_df, valid_groups, save_path)
        else:
            return self._plot_generic_patterns(anova_results, filtered_df, valid_groups, save_path)
    
    def _plot_success_rate_patterns(self, anova_results, filtered_df, valid_groups, save_path):
        """Plot success rate patterns by age group - FIXED VERSION"""
        
        fig, axes = plt.subplots(1, len(valid_groups), figsize=(5*len(valid_groups), 6))
        if len(valid_groups) == 1:
            axes = [axes]
        
        trial_order = ['vis1', 'invis', 'vis2']
        trial_colors = [self.colors.get('vis1', '#1f77b4'), 
                       self.colors.get('invis', '#ff7f0e'), 
                       self.colors.get('vis2', '#2ca02c')]
        
        # Get age group definitions
        age_groups = anova_results.get('age_group_definitions', {})
        
        for i, group in enumerate(valid_groups):
            ax = axes[i]
            
            if group not in age_groups:
                continue
            
            min_age, max_age = age_groups[group]
            
            # Filter data for this age group
            age_mask = (filtered_df['age'] >= min_age) & (filtered_df['age'] < max_age)
            group_df = filtered_df[age_mask]
            
            # Calculate mean success rates for each trial type
            trial_means = []
            trial_stds = []
            trial_labels = []
            
            for trial in trial_order:
                # Combine max and min conditions
                trial_data = []
                for condition in ['max', 'min']:
                    col = f'{trial}_sr_{condition}_const'
                    if col in group_df.columns:
                        data = group_df[col].dropna()
                        trial_data.extend(data.values)
                
                if trial_data:
                    trial_means.append(np.mean(trial_data))
                    trial_stds.append(np.std(trial_data))
                    trial_labels.append(trial.upper())
                else:
                    trial_means.append(np.nan)
                    trial_stds.append(np.nan)
                    trial_labels.append(trial.upper())
            
            # Remove NaN values for plotting
            valid_indices = [j for j, val in enumerate(trial_means) if not np.isnan(val)]
            valid_trials = [trial_labels[j] for j in valid_indices]
            valid_means = [trial_means[j] for j in valid_indices]
            valid_stds = [trial_stds[j] for j in valid_indices]
            valid_colors = [trial_colors[j] for j in valid_indices]
            
            if valid_means:
                # Plot with error bars
                x_pos = np.arange(len(valid_trials))
                bars = ax.bar(x_pos, valid_means, yerr=valid_stds, 
                             color=valid_colors, alpha=0.7, capsize=5,
                             edgecolor='black', linewidth=1)
                
                # Add value labels
                for j, (bar, mean, std) in enumerate(zip(bars, valid_means, valid_stds)):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height + std + 0.02,
                           f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')
                
                ax.set_xticks(x_pos)
                ax.set_xticklabels(valid_trials)
                ax.set_ylim(0, 1.1)
            
            # Group info
            group_data = anova_results['group_analyses'][group]
            n_subjects = group_data['descriptive_stats']['n_subjects']
            mean_age = anova_results['group_assignments'][group]['mean_age']
            
            ax.set_ylabel('Success Rate')
            ax.set_title(f'{group.title()}\n(n={n_subjects}, age={mean_age:.1f}y)')
            ax.grid(True, alpha=0.3)
            
            # Add significance annotation
            if 'trial_type_effect' in group_data:
                sig = group_data['trial_type_effect']['significant']
                p_val = group_data['trial_type_effect']['p_value']
                sig_text = f"Trial effect: {'Yes' if sig else 'No'}\np = {p_val:.3f}"
                
                ax.text(0.05, 0.95, sig_text, transform=ax.transAxes, 
                       bbox=dict(boxstyle='round', 
                                facecolor='lightgreen' if sig else 'lightcoral', 
                                alpha=0.7),
                       verticalalignment='top', fontsize=9)
        
        plt.suptitle(f'{anova_results["dependent_variable"]} Patterns by Age Group', 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Success rate patterns saved to: {save_path}")
        
        return fig
    

    def _plot_learning_patterns(self, anova_results, filtered_df, valid_groups, save_path):
        """Plot learning metric patterns by age group"""
        print(f"Learning patterns visualization not yet implemented. Using generic pattern plot.")
        return self._plot_generic_patterns(anova_results, filtered_df, valid_groups, save_path)
    
    def _plot_retention_patterns(self, anova_results, filtered_df, valid_groups, save_path):
        """Plot retention patterns by age group"""
        print(f"Retention patterns visualization not yet implemented. Using generic pattern plot.")
        return self._plot_generic_patterns(anova_results, filtered_df, valid_groups, save_path)
    
    def _plot_stride_patterns(self, anova_results, filtered_df, valid_groups, save_path):
        """Plot stride-related patterns by age group"""
        print(f"Stride patterns visualization not yet implemented. Using generic pattern plot.")
        return self._plot_generic_patterns(anova_results, filtered_df, valid_groups, save_path)
    
    def _plot_generic_patterns(self, anova_results, filtered_df, valid_groups, save_path):
        """Generic pattern plot for any dependent variable"""
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        # Simple bar plot of effect sizes
        effect_sizes = []
        group_names = []
        
        for group in valid_groups:
            group_data = anova_results['group_analyses'][group]
            if 'trial_type_effect' in group_data:
                effect_sizes.append(group_data['trial_type_effect']['effect_size_eta2'])
                group_names.append(group.title())
        
        if effect_sizes:
            bars = ax.bar(group_names, effect_sizes, alpha=0.7, 
                         color=self.colors.get('primary', '#667eea'))
            
            for bar, eta2 in zip(bars, effect_sizes):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                       f'{eta2:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax.set_ylabel('Effect Size (η²)')
        ax.set_title(f'{anova_results["dependent_variable"]} Effect Sizes by Age Group')
        ax.grid(True, alpha=0.3)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Generic patterns saved to: {save_path}")
        
        return fig

In [ ]:
# Integration example with your existing motor learning analysis pipeline

# After running your main analysis setup:
data_root_dir = 'muh_data/'
metadata_path = 'muh_metadata_tape.csv'

# Your existing analysis setup
analysis = (MotorLearningAnalysis()
    .load_data(metadata_path, data_root_dir, force_reprocess=True)
    .filter_data(required_trial_types=['vis1', 'invis', 'vis2'])
    .calculate_metrics())

# Apply motor noise filter
if 'mot_noise' in analysis.metrics_df.columns:
    motor_noise_filtered = analysis.metrics_df[analysis.metrics_df['mot_noise'] <= 0.15]
    analysis.metrics_df = motor_noise_filtered
    analysis.statistical_analyzer = StatisticalAnalyzer(motor_noise_filtered, analysis.config)
    analysis.population_visualizer = PopulationVisualizer(motor_noise_filtered, analysis.config)

# Get the components
stats = analysis.statistical_analyzer
pop_viz = analysis.population_visualizer
metrics_df = analysis.metrics_df

# ============================================================================
# ENHANCED ANOVA VISUALIZATIONS
# ============================================================================

# 1. Create the enhanced visualizer
from pathlib import Path
anova_viz = AgeStratifiedANOVAVisualizer(pop_viz.config, pop_viz.colors)

# 2. Run comprehensive age-stratified analyses
print("Running comprehensive age-stratified ANOVA analyses...")

# Define custom age groups if needed
custom_age_groups = {
    'younger': [7, 12],
    'middle': [12, 15], 
    'older': [15, 18]
}

# Different dependent variables to analyze
dependent_variables = {
    'Success Rate': '_sr_',
    'Learning': '_learning_', 
    'Retention': '_retention_',  # This will now work correctly
    'Mean Stride Length': '_msl_',
    'Stride Variability': '_sd_'
}

# Run analyses for each dependent variable
anova_results = {}
for var_name, var_pattern in dependent_variables.items():
    print(f"\nAnalyzing {var_name}...")
    
    results = stats.run_unified_age_stratified_anova(
        dependent_var_pattern=var_pattern,
        dependent_var_name=var_name,
        age_groups=custom_age_groups,
        min_subjects_per_group=5  # Minimum subjects per age group
    )
    
    if 'error' not in results:
        anova_results[var_name] = results
        print(f"  ✓ Successfully analyzed {var_name}")
        
        # Print quick summary
        valid_groups = [name for name, group_results in results['group_analyses'].items() 
                       if 'error' not in group_results]
        print(f"  - Valid age groups: {valid_groups}")
        print(f"  - Total subjects: {results['total_subjects_analyzed']}")
    else:
        print(f"  ✗ Error in {var_name}: {results['error']}")

# 3. Create comprehensive visualizations
print("\n" + "="*60)
print("CREATING ENHANCED ANOVA VISUALIZATIONS")
print("="*60)

# Create output directory for ANOVA plots
anova_output_dir = analysis.config.figures_dir / 'anova_analyses'
anova_output_dir.mkdir(exist_ok=True)

for var_name, results in anova_results.items():
    print(f"\nCreating visualizations for {var_name}...")
    
    # 1. Comprehensive ANOVA summary (4-panel plot)
    summary_path = anova_output_dir / f'anova_summary_{var_name.lower().replace(" ", "_")}.png'
    fig1 = anova_viz.plot_unified_anova_summary(results, summary_path)
    
    # 2. Dependent variable patterns by age group
    patterns_path = anova_output_dir / f'patterns_{var_name.lower().replace(" ", "_")}_by_age.png'
    fig2 = anova_viz.plot_dependent_variable_patterns(results, metrics_df, patterns_path)
    
    if fig1:
        print(f"  ✓ Created summary plot: {summary_path.name}")
    if fig2:
        print(f"  ✓ Created patterns plot: {patterns_path.name}")

# 4. Create comparative visualization across all measures
if len(anova_results) > 1:
    print(f"\nCreating comparative visualization across all measures...")
    comparative_path = anova_output_dir / 'comparative_anova_effects.png'
    fig_comp = create_comparative_anova_plot(anova_results)

In [ ]:
# Corrected usage example for your motor learning analysis
from pathlib import Path

data_root_dir = 'muh_data/'
metadata_path = 'muh_metadata_tape.csv'

# Step-by-step analysis (recommended approach)
analysis = (MotorLearningAnalysis()
    .load_data(metadata_path, data_root_dir, force_reprocess=False)
    .filter_data(required_trial_types=['vis1', 'invis', 'vis2'])
    .calculate_metrics())

# Get initial metrics before motor noise filtering
initial_metrics = analysis.metrics_df
print(f"Initial dataset: {len(initial_metrics)} subjects")
print(f"Subjects with motor noise data: {initial_metrics['mot_noise'].notna().sum()}")

# Apply motor noise filter (≤ 0.3)
if 'mot_noise' in initial_metrics.columns:
    motor_noise_filtered = initial_metrics[initial_metrics['mot_noise'] <= 0.15]
    print(f"After motor noise filter (≤ 0.3): {len(motor_noise_filtered)} subjects")
    print(f"Subjects excluded due to high motor noise: {len(initial_metrics) - len(motor_noise_filtered)}")
    
    # Update the analysis with filtered data
    analysis.metrics_df = motor_noise_filtered
    
    # Re-initialize analyzers with filtered data
    analysis.statistical_analyzer = StatisticalAnalyzer(motor_noise_filtered, analysis.config)
    analysis.population_visualizer = PopulationVisualizer(motor_noise_filtered, analysis.config)
    # Individual visualizer uses data_manager, so no need to update
else:
    print("No motor noise data found - proceeding without motor noise filter")
    motor_noise_filtered = initial_metrics

# Now you can access all components (with filtered data)
data_manager = analysis.data_manager
metrics_df = analysis.metrics_df  # This is now the filtered dataset
stats = analysis.statistical_analyzer
pop_viz = analysis.population_visualizer
ind_viz = analysis.individual_visualizer

# Run specific analyses
print("Running regression analysis...")
regression_results = stats.run_regression_analysis()

print("Running correlation analysis...")
correlation_results = stats.run_correlation_analysis()

# Run full analysis to generate visualizations and report
print("Running full analysis...")
analysis.run_analysis()

# Access results
full_results = analysis.get_results()

# Print summary
print("\n" + "="*60)
print("ANALYSIS SUMMARY")
print("="*60)

print(f"\nFiltered Dataset Info:")
print(f"  Total subjects (after motor noise filter): {len(metrics_df)}")
print(f"  Subjects with motor noise data: {metrics_df['mot_noise'].notna().sum()}")
if 'mot_noise' in metrics_df.columns:
    print(f"  Motor noise range: {metrics_df['mot_noise'].min():.3f} - {metrics_df['mot_noise'].max():.3f}")
    print(f"  Mean motor noise: {metrics_df['mot_noise'].mean():.3f}")
print(f"  Age range: {metrics_df['age'].min():.1f} - {metrics_df['age'].max():.1f} years")

print(f"\nRegression Results:")
print(f"  R²: {regression_results['metrics']['r2']:.3f}")
print(f"  RMSE: {regression_results['metrics']['rmse']:.3f}")
print(f"  Sample size: {regression_results['metrics']['n_samples']}")
print(f"  Feature importances:")
for feature, importance in regression_results['feature_importances'].items():
    print(f"    {feature}: {importance:.3f}")


print(f"\nSignificant Correlations:")
sig_corrs = correlation_results.get('significant_correlations', {})
if sig_corrs:
    for corr_name, corr_data in sig_corrs.items():
        print(f"  {corr_name}: r={corr_data['r']:.3f}, p={corr_data['p']:.3f}, n={corr_data['n']}")
else:
    print("  No significant correlations found")

print(f"\nOutput Directory: {analysis.config.base_output_dir}")
print(f"  - Figures: {analysis.config.figures_dir}")
print(f"    - Population plots (including ANOVA visualizations)")
print(f"    - Individual subject plots")
print(f"  - Reports: {analysis.config.reports_dir}")
print(f"  - Processed Data: {analysis.config.processed_data_dir}")

# List specific plots created
if 'visualizations' in full_results:
    pop_plots = full_results['visualizations'].get('population', [])
    if pop_plots:
        print(f"\nPopulation plots created:")
        for plot in pop_plots:
            plot_name = Path(plot).name
            print(f"  - {plot_name}")
    
    ind_plots = full_results['visualizations'].get('individual', [])
    if ind_plots:
        print(f"\nIndividual plots created:")
        for plot in ind_plots:
            plot_name = Path(plot).name
            print(f"  - {plot_name}")

# Generate individual plots for a few subjects
print(f"\nGenerating individual subject plots...")
subject_ids = list(data_manager.subjects.keys())[:3]
for subject_id in subject_ids:
    try:
        plot_path = ind_viz.plot_subject_summary(subject_id)
        if plot_path:
            print(f"  Created summary for {subject_id}: {plot_path}")
    except Exception as e:
        print(f"  Failed to create plot for {subject_id}: {e}")

print(f"\nAnalysis complete! Check the output directory for all results.")

In [ ]:
def plot_all_subjects_stride_changes_by_age(pop_viz, subjects_per_plot=5):
    """
    Create stride change distribution plots for all subjects, organized by age.
    
    Parameters:
    -----------
    pop_viz : PopulationVisualizer
        The population visualizer instance
    subjects_per_plot : int, default=5
        Number of subjects to include in each plot
    
    Returns:
    --------
    list : List of paths to created plots
    """
    
    # Sort subjects by age
    df_sorted = pop_viz.filtered_df.sort_values('age').reset_index(drop=True)
    total_subjects = len(df_sorted)
    plot_paths = []
    
    print(f"Creating stride change plots for {total_subjects} subjects (sorted by age, {subjects_per_plot} per plot)")
    print(f"Age range: {df_sorted['age'].min():.1f} - {df_sorted['age'].max():.1f} years")
    
    # Create plots in batches
    for start_idx in range(0, total_subjects, subjects_per_plot):
        end_idx = min(start_idx + subjects_per_plot, total_subjects)
        batch_subjects = df_sorted.iloc[start_idx:end_idx]
        batch_num = (start_idx // subjects_per_plot) + 1
        
        # Get age range for this batch
        min_age = batch_subjects['age'].min()
        max_age = batch_subjects['age'].max()
        
        print(f"Creating batch {batch_num}: subjects {start_idx + 1}-{end_idx}, ages {min_age:.1f}-{max_age:.1f}y")
        
        # Temporarily replace the dataframe with sorted batch
        original_df = pop_viz.filtered_df
        pop_viz.filtered_df = batch_subjects
        
        try:
            # Create plot for this batch
            plot_path = pop_viz.plot_individual_stride_change_distributions(
                max_subjects=subjects_per_plot
            )
            
            if plot_path and plot_path.exists():
                # Create new filename with age info
                new_name = f'stride_changes_by_age_batch_{batch_num:02d}_ages_{min_age:.1f}-{max_age:.1f}y.png'
                new_path = plot_path.parent / new_name
                
                # Delete target file if it exists
                if new_path.exists():
                    try:
                        new_path.unlink()
                        print(f"  Deleted existing: {new_name}")
                    except Exception as del_e:
                        print(f"  Warning: Could not delete existing file: {del_e}")
                
                # Rename the file
                try:
                    plot_path.rename(new_path)
                    plot_paths.append(new_path)
                    print(f"  Saved: {new_name}")
                except Exception as rename_e:
                    print(f"  Error renaming file: {rename_e}")
                    # If rename fails, try copying instead
                    try:
                        import shutil
                        shutil.copy2(plot_path, new_path)
                        plot_path.unlink()  # Delete original
                        plot_paths.append(new_path)
                        print(f"  Copied and saved: {new_name}")
                    except Exception as copy_e:
                        print(f"  Error copying file: {copy_e}")
            else:
                print(f"  No plot created for batch {batch_num}")
                
        except Exception as e:
            print(f"  Error creating batch {batch_num}: {e}")
        finally:
            # Restore original dataframe
            pop_viz.filtered_df = original_df
    
    print(f"\nCompleted: Created {len(plot_paths)} age-organized plots in {pop_viz.config.population_plots_dir}")
    return plot_paths


# Simple usage function
def create_stride_change_plots_by_age(analysis, subjects_per_plot=5):
    """
    Create stride change plots sorted by age.
    
    Usage:
    ------
    plot_paths = create_stride_change_plots_by_age(analysis, subjects_per_plot=5)
    """
    return plot_all_subjects_stride_changes_by_age(analysis.population_visualizer, subjects_per_plot)

create_stride_change_plots_by_age(analysis, subjects_per_plot=5)


In [ ]:
results = stats.run_unified_age_stratified_anova('_retention_')

results

In [ ]:
# After running your analysis
pop_viz = analysis.population_visualizer

# 1. Individual participant distributions (first 20 subjects)
individual_plot = pop_viz.plot_individual_stride_change_distributions(max_subjects=20)

# 2. Age group combined distributions
age_group_plot = pop_viz.plot_age_group_stride_change_distributions()

# 3. Summary comparison across age groups
summary_plot = pop_viz.plot_stride_change_comparison_summary()

print(f"Plots saved:")
print(f"  Individual: {individual_plot}")
print(f"  Age groups: {age_group_plot}")
print(f"  Summary: {summary_plot}")

In [ ]:
def diagnose_retention_calculation(analysis):
    """
    Comprehensive diagnostic tool for retention calculation issues.
    
    Parameters:
    -----------
    analysis : MotorLearningAnalysis
        Your analysis object with data_manager and metrics_df
    
    Returns:
    --------
    Dict with diagnostic information
    """
    
    diagnostics = {
        'column_check': {},
        'subject_analysis': {},
        'data_availability': {},
        'calculation_issues': []
    }
    
    data_manager = analysis.data_manager
    metrics_df = analysis.metrics_df
    
    # 1. CHECK COLUMN EXISTENCE
    print("="*60)
    print("RETENTION CALCULATION DIAGNOSTICS")
    print("="*60)
    
    retention_columns = [col for col in metrics_df.columns if 'retention' in col]
    print(f"\n1. RETENTION COLUMNS FOUND: {len(retention_columns)}")
    for col in retention_columns:
        non_null_count = metrics_df[col].notna().sum()
        total_count = len(metrics_df)
        print(f"   - {col}: {non_null_count}/{total_count} subjects ({non_null_count/total_count*100:.1f}%)")
        diagnostics['column_check'][col] = {
            'non_null': non_null_count,
            'total': total_count,
            'percentage': non_null_count/total_count*100
        }
    
    if not retention_columns:
        print("   ❌ NO RETENTION COLUMNS FOUND!")
        return diagnostics
    
    # 2. ANALYZE INDIVIDUAL SUBJECTS
    print(f"\n2. INDIVIDUAL SUBJECT ANALYSIS:")
    print(f"   Checking first 10 subjects with invisible trial data...")
    
    subjects_analyzed = 0
    subjects_with_retention = 0
    
    for subject_id, subject in data_manager.subjects.items():
        if subjects_analyzed >= 10:
            break
            
        invis_trial = subject.trial_data.get('invis')
        if not invis_trial or invis_trial['data'] is None:
            continue
            
        subjects_analyzed += 1
        df = invis_trial['data']
        
        print(f"\n   Subject {subject_id}:")
        print(f"     - Age: {subject.age:.1f} years")
        print(f"     - Invisible trial strides: {len(df)}")
        
        # Check for required columns
        required_cols = ['Target size', 'Constant', 'Sum of gains and steps', 'Stride Number']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"     ❌ Missing columns: {missing_cols}")
            continue
            
        # Analyze clamp periods
        max_target = df['Target size'].max()
        clamp_mask = (
            (df['Target size'] >= max_target - 0.01) & 
            (np.abs(df['Constant'] - 2.0) < 0.01)
        )
        clamp_data = df[clamp_mask]
        
        if clamp_data.empty:
            print(f"     ❌ No success clamp periods found (max_target={max_target:.3f})")
            diagnostics['calculation_issues'].append(f"{subject_id}: No clamp periods")
            continue
            
        print(f"     - Max target size: {max_target:.3f}")
        print(f"     - Clamp strides found: {len(clamp_data)}")
        
        # Group consecutive clamp strides
        clamp_data_sorted = clamp_data.sort_values('Stride Number').copy()
        clamp_data_sorted['stride_diff'] = clamp_data_sorted['Stride Number'].diff()
        clamp_data_sorted['new_period'] = (clamp_data_sorted['stride_diff'] > 1).fillna(True)
        clamp_data_sorted['period_id'] = clamp_data_sorted['new_period'].cumsum()
        
        # Find periods with sufficient strides
        period_info = []
        for period_id in clamp_data_sorted['period_id'].unique():
            period_strides = clamp_data_sorted[clamp_data_sorted['period_id'] == period_id]
            if len(period_strides) >= 15:  # Minimum threshold from code
                period_info.append({
                    'period_id': period_id,
                    'start': period_strides['Stride Number'].min(),
                    'end': period_strides['Stride Number'].max(),
                    'length': len(period_strides)
                })
        
        print(f"     - Valid clamp periods: {len(period_info)}")
        for i, period in enumerate(period_info):
            print(f"       Period {i+1}: Strides {period['start']}-{period['end']} (n={period['length']})")
            
            # Check for pre-clamp learning data
            search_start = max(1, period['start'] - 50)
            search_end = period['start'] - 1
            learning_mask = (
                (df['Stride Number'] >= search_start) & 
                (df['Stride Number'] <= search_end) &
                (df['Target size'] < max_target - 0.01)
            )
            learning_data = df[learning_mask]
            
            if len(learning_data) >= 10:
                recent_learning = learning_data.tail(15)
                pre_clamp_performance = recent_learning['Sum of gains and steps'].mean()
                print(f"         - Pre-clamp learning available: {len(learning_data)} strides")
                print(f"         - Pre-clamp performance: {pre_clamp_performance:.3f}")
            else:
                print(f"         - ❌ Insufficient pre-clamp learning: {len(learning_data)} strides")
        
        if len(period_info) > 0:
            subjects_with_retention += 1
    
    print(f"\n   Summary: {subjects_with_retention}/{subjects_analyzed} subjects have potential for retention calculation")
    
    # 3. DATA AVAILABILITY SUMMARY
    print(f"\n3. DATA AVAILABILITY SUMMARY:")
    
    total_subjects = len(data_manager.subjects)
    subjects_with_invis = sum(1 for s in data_manager.subjects.values() 
                             if 'invis' in s.trial_data and s.trial_data['invis']['data'] is not None)
    
    print(f"   - Total subjects: {total_subjects}")
    print(f"   - Subjects with invisible trials: {subjects_with_invis}")
    
    diagnostics['data_availability'] = {
        'total_subjects': total_subjects,
        'subjects_with_invis': subjects_with_invis,
        'subjects_analyzed': subjects_analyzed,
        'subjects_with_potential_retention': subjects_with_retention
    }
    
    # 4. RECOMMEND SOLUTIONS
    print(f"\n4. RECOMMENDED SOLUTIONS:")
    
    if not retention_columns:
        print("   🔧 CRITICAL: Retention columns not being created properly")
        print("      - Check _calculate_retention_for_all_subjects() method")
        print("      - Ensure column names match between creation and lookup")
    
    if subjects_with_retention < subjects_analyzed * 0.5:
        print("   🔧 Many subjects lack retention data due to:")
        print("      - Insufficient clamp periods (need ≥15 strides)")
        print("      - Missing pre-clamp learning data")
        print("      - Consider relaxing criteria:")
        print("        * Reduce minimum clamp period to 10 strides")
        print("        * Reduce pre-clamp learning requirement to 5 strides")
        print("        * Allow wider tolerance for target size matching")
    
    print(f"\n5. DEBUGGING STEPS:")
    print("   1. Add debug prints in _calculate_subject_retention()")
    print("   2. Check if metrics are being properly stored")
    print("   3. Verify column name consistency")
    print("   4. Consider manual calculation for one subject to verify logic")
    
    return diagnostics

# USAGE EXAMPLE:
# diagnostics = diagnose_retention_calculation(analysis)

def fix_retention_calculation(analysis):
    """
    Apply fixes to retention calculation issues.
    """
    print("Applying retention calculation fixes...")
    
    # Get the data manager
    data_manager = analysis.data_manager
    
    # Re-run retention calculation with debug output
    print("Re-calculating retention metrics with debug output...")
    
    retention_results = []
    
    for subject_id, subject in data_manager.subjects.items():
        invis_trial = subject.trial_data.get('invis')
        if not invis_trial or invis_trial['data'] is None:
            continue
            
        print(f"\nProcessing {subject_id}...")
        
        # Use the existing retention calculation but with debug
        retention_metrics = data_manager.metrics_engine._calculate_subject_retention(
            subject_id, invis_trial['data']
        )
        
        if any(not pd.isna(val) for val in retention_metrics.values()):
            retention_results.append({
                'subject_id': subject_id,
                **retention_metrics
            })
            print(f"  ✓ Calculated retention for {subject_id}")
            for metric, value in retention_metrics.items():
                if not pd.isna(value):
                    print(f"    {metric}: {value:.1f}%")
        else:
            print(f"  ❌ No retention data for {subject_id}")
    
    print(f"\nSuccessfully calculated retention for {len(retention_results)} subjects")
    
    return retention_results

# USAGE:
# retention_data = fix_retention_calculation(analysis)

In [ ]:
# Import the diagnostic function and run it
diagnostics = diagnose_retention_calculation(analysis)